In [ ]:
%%capture
from __future__ import annotations
# Core imports
import os
import sys
import json
import csv
import re
from _library.runtime import configure_runtime, LOW_POWER
configure_runtime()
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, field, asdict
from datetime import datetime
from collections import Counter
from html import escape
import warnings
import base64
warnings.filterwarnings('ignore')

# Display and widgets
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets
from ipywidgets import Layout, Button, Box, VBox, HBox, Label, HTML as HTMLWidget

# Optional visualization
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    VISUALIZATION_AVAILABLE = True
except ImportError:
    plt = None
    sns = None
    VISUALIZATION_AVAILABLE = False

# Progress tracking
from tqdm.auto import tqdm

# Notebook/bootstrap paths
def resolve_notebook_root(start: Optional[Path] = None) -> Path:
    candidates = []

    env_root = os.environ.get('NSDDD_NOTEBOOK_ROOT')
    if env_root:
        candidates.append(Path(env_root).expanduser())

    if start is not None:
        candidates.append(start)
    candidates.append(Path.cwd())

    session_name = os.environ.get('JPY_SESSION_NAME')
    if session_name:
        session_path = Path(session_name).expanduser()
        if session_path.is_absolute():
            candidates.append(session_path.parent if session_path.suffix else session_path)

    if '__file__' in globals():
        candidates.append(Path(__file__).resolve().parent)

    seen = set()
    for base in candidates:
        try:
            base_resolved = base.resolve()
        except Exception:
            continue
        for candidate in [base_resolved, *base_resolved.parents]:
            if str(candidate) in seen:
                continue
            seen.add(str(candidate))
            if (candidate / '_library').exists():
                return candidate

    return Path.cwd().resolve()

NOTEBOOK_ROOT = resolve_notebook_root(Path.cwd())
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

# Country lookup
from _library.countries import CountryLookup

# Optional: Fuzzy matching (for keyword search)
try:
    from rapidfuzz import fuzz              # fast, and installed with the app
    FUZZY_AVAILABLE = True
except ImportError:
    try:
        from fuzzywuzzy import fuzz         # older package with the same partial_ratio function (slower)
        FUZZY_AVAILABLE = True
    except ImportError:
        FUZZY_AVAILABLE = False

print('✅ Imports loaded successfully')
if not VISUALIZATION_AVAILABLE:
    print('⚠️  Charts disabled (matplotlib/seaborn not installed)')
if FUZZY_AVAILABLE:
    print('✅ Fuzzy matching available')
else:
    print('⚠️  Fuzzy matching not available (install rapidfuzz for fuzzy keyword matching)')
print(f'📅 Initialized: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

def inject_app_favicon():
    """Inject a custom SVG favicon for browser tabs (Voila/Jupyter)."""
    svg = '''<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 64 64">
  <defs>
    <linearGradient id="bg" x1="0" y1="0" x2="1" y2="1">
      <stop offset="0%" stop-color="#0f2740"/>
      <stop offset="100%" stop-color="#1f3f5c"/>
    </linearGradient>
  </defs>
  <rect x="0" y="0" width="64" height="64" rx="12" fill="url(#bg)"/>
  <circle cx="32" cy="32" r="24" fill="none" stroke="#4fc3f7" stroke-width="4.8" stroke-dasharray="20 9" transform="rotate(-20 32 32)"/>
  <circle cx="32" cy="32" r="17" fill="none" stroke="#ff7043" stroke-width="4.6" stroke-dasharray="17 8" transform="rotate(18 32 32)"/>
  <circle cx="32" cy="32" r="11" fill="none" stroke="#ffd54f" stroke-width="3.4" stroke-dasharray="12 6" transform="rotate(-10 32 32)"/>
  <rect x="20" y="14" width="24" height="34" rx="2.6" fill="#ffffff" stroke="#263238" stroke-width="1.8"/>
  <line x1="25" y1="21" x2="39" y2="21" stroke="#455a64" stroke-width="1.8"/>
  <line x1="25" y1="27" x2="39" y2="27" stroke="#455a64" stroke-width="1.8"/>
  <line x1="25" y1="33" x2="39" y2="33" stroke="#455a64" stroke-width="1.8"/>
  <line x1="25" y1="39" x2="39" y2="39" stroke="#455a64" stroke-width="1.8"/>
  <circle cx="10" cy="12" r="1.6" fill="#80deea"/>
  <circle cx="52" cy="9" r="1.4" fill="#ffca28"/>
  <circle cx="55" cy="53" r="1.4" fill="#ef9a9a"/>
</svg>'''

    data_uri = 'data:image/svg+xml;base64,' + base64.b64encode(svg.encode('utf-8')).decode('ascii')
    script = f'''<script>
(function() {{
  var href = "{data_uri}";
  var existing = document.querySelector("link[rel='icon']") || document.querySelector("link[rel='shortcut icon']");
  if (!existing) {{
    existing = document.createElement('link');
    existing.setAttribute('rel', 'icon');
    document.head.appendChild(existing);
  }}
  existing.setAttribute('type', 'image/svg+xml');
  existing.setAttribute('href', href);
}})();
</script>'''
    display(HTML(script))



In [ ]:
%%capture
# Define paths (supports installer, configured install paths, and legacy repo layouts)
ROOT_PATH = Path.cwd()
INSTALL_CONFIG_PATH = ROOT_PATH / 'install_config.json'
CONFIGURED_INSTALL_PATH = None

if INSTALL_CONFIG_PATH.exists():
    try:
        _install_config = json.loads(INSTALL_CONFIG_PATH.read_text(encoding='utf-8'))
        _configured = _install_config.get('install_path')
        if _configured:
            from _library.runtime import resolve_path
            CONFIGURED_INSTALL_PATH = resolve_path(_configured, ROOT_PATH)
    except Exception:
        CONFIGURED_INSTALL_PATH = None

if os.environ.get('NSDDD_INSTALL_PATH'):
    from _library.runtime import resolve_path
    CONFIGURED_INSTALL_PATH = resolve_path(os.environ['NSDDD_INSTALL_PATH'], ROOT_PATH)

BASE_CANDIDATES = []
MODEL_CANDIDATES = []

if CONFIGURED_INSTALL_PATH is not None:
    BASE_CANDIDATES.append(CONFIGURED_INSTALL_PATH / 'metadata')
    MODEL_CANDIDATES.append(CONFIGURED_INSTALL_PATH / 'model')

BASE_CANDIDATES.extend([
    ROOT_PATH / 'NSDDD_v4.0_workspace' / 'metadata',
    ROOT_PATH / 'NSDDD_v3.5_workspace' / 'metadata',
    ROOT_PATH / 'NSDDD_v3_workspace' / 'metadata',
    ROOT_PATH / 'metadata',
    ROOT_PATH / '..' / 'document_processing' / 'data' / 'metadata',
])
MODEL_CANDIDATES.extend([
    ROOT_PATH / 'NSDDD_v4.0_workspace' / 'model',
    ROOT_PATH / 'NSDDD_v3.5_workspace' / 'model',
    ROOT_PATH / 'NSDDD_v3_workspace' / 'model',
    ROOT_PATH / 'model',
    ROOT_PATH / '..' / 'model',
])

BASE_PATH = next((p for p in BASE_CANDIDATES if p.exists()), BASE_CANDIDATES[0])
MODEL_PATH = next((p for p in MODEL_CANDIDATES if p.exists()), MODEL_CANDIDATES[0])

if not (BASE_PATH / 'document_metadata.csv').exists() and (BASE_PATH / 'metadata' / 'document_metadata.csv').exists():
    BASE_PATH = BASE_PATH / 'metadata'

print(f'📁 Metadata path: {BASE_PATH.resolve()}')
print(f'📁 Model path   : {MODEL_PATH.resolve()}')
if CONFIGURED_INSTALL_PATH is not None:
    print(f'📁 Install path : {CONFIGURED_INSTALL_PATH.resolve()}')
if not BASE_PATH.exists():
    print('⚠️  Metadata path not found. Filters may be empty until metadata is installed.')
if not MODEL_PATH.exists():
    print('⚠️  Model path not found. Search model will not load until installation completes.')

LOCAL_VERSION_CANDIDATES = [
    ROOT_PATH / 'VERSION',
    ROOT_PATH.parent / 'VERSION',
]
LOCAL_VERSION_PATH = next((p for p in LOCAL_VERSION_CANDIDATES if p.exists()), LOCAL_VERSION_CANDIDATES[0])
REMOTE_VERSION_URL = 'https://raw.githubusercontent.com/andrewneal78/NSDDD_v4.0_installer/main/VERSION'
NOTEBOOK_UPDATE_INFO = None


def _read_local_version() -> str:
    try:
        return LOCAL_VERSION_PATH.read_text(encoding='utf-8').strip()
    except Exception:
        return 'unknown'


def _version_key(value: str):
    parts = []
    for chunk in str(value).strip().split('.'):
        try:
            parts.append(int(chunk))
        except ValueError:
            parts.append(chunk)
    return tuple(parts)


def check_notebook_update() -> Dict[str, Optional[str]]:
    info = {
        'local': _read_local_version(),
        'remote': None,
        'error': None,
        'update_available': False,
    }

    if (os.environ.get('NSDDD_AUTO_UPDATE') == '0' or
            any(os.environ.get(k, '').lower() in ('1', 'true', 'yes')
                for k in ('HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE'))):
        return info

    try:
        import urllib.request

        with urllib.request.urlopen(REMOTE_VERSION_URL, timeout=3) as response:
            remote_version = response.read().decode('utf-8').strip()
        info['remote'] = remote_version
        info['update_available'] = _version_key(remote_version) > _version_key(info['local'])
    except Exception as exc:
        info['error'] = str(exc)

    return info


def _version_key(v) -> list:
    return [int(p) if p.isdigit() else 0 for p in str(v or '').strip().split('.')]


def _changes_between(before, after) -> list:
    """Entries of CHANGELOG.json newer than `before` and not newer than `after`, newest first (latest three if `before` is unknown)."""
    try:
        entries = json.loads((Path.cwd() / 'CHANGELOG.json').read_text(encoding='utf-8'))
    except Exception:
        return []
    picked = [e for e in entries if e.get('version') and _version_key(e['version']) <= _version_key(after or '9999')
              and (not before or _version_key(e['version']) > _version_key(before))]
    return picked if before else picked[:3]


def _updated_notice_html() -> str:
    """Shown once after launch.py installed a program and/or data update when NSDDD started: lists what changed."""
    path = Path.cwd() / 'update_notice.json'
    try:
        info = json.loads(path.read_text(encoding='utf-8'))
        path.unlink()
    except Exception:
        return ''
    sections = []
    if info.get('to'):
        head = (f'Program updated from version <code>{escape(str(info.get("from") or "unknown"))}</code> to '
                f'<code>{escape(str(info["to"]))}</code>.')
        entries = _changes_between(info.get('from'), info['to'])
        body = ''
        for e in entries[:6]:
            items = ''.join(f'<li>{escape(str(c))}</li>' for c in e.get('changes', []))
            label = f'<div style="margin-top: 6px;"><strong>Version {escape(str(e["version"]))}</strong>' + (
                f' ({escape(str(e["date"]))})' if e.get('date') else '') + '</div>' if len(entries) > 1 else ''
            body += label + f'<ul style="margin: 4px 0 0 0; padding-left: 20px;">{items}</ul>'
        sections.append(f'<div style="margin-top: 8px;"><strong>Program</strong><br>{head}{body}</div>')
    data = info.get('data') or {}
    if data.get('version'):
        added = data.get('documents_added') or []
        removed = data.get('documents_removed') or []
        countries = data.get('countries_added') or []
        parts = [f'New data installed (data version <code>{escape(str(data["version"]))}</code>).']
        if data.get('notes'):
            parts.append(escape(str(data['notes'])))
        body = ''
        if added:
            shown = ''.join(
                f'<li>{escape(str(d.get("title", "")))}' + (f' ({escape(str(d["type"]))})' if d.get('type') else '') + '</li>'
                for d in added[:25])
            more = f'<li>and {len(added) - 25} more</li>' if len(added) > 25 else ''
            body += (f'<div style="margin-top: 6px;"><strong>{len(added)} new document{"s" if len(added) != 1 else ""}</strong>'
                     f'</div><ul style="margin: 4px 0 0 0; padding-left: 20px;">{shown}{more}</ul>')
        if countries:
            body += f'<div style="margin-top: 6px;"><strong>New countries:</strong> {escape(", ".join(countries))}</div>'
        if removed:
            body += (f'<div style="margin-top: 6px;"><strong>{len(removed)} document{"s" if len(removed) != 1 else ""} removed:</strong> '
                     + escape('; '.join(str(d.get("title", "")) for d in removed[:10])) + ('; ...' if len(removed) > 10 else '') + '</div>')
        if not (added or removed or countries) and 'documents_added' in data:    # only when the old and new lists were compared
            body += ('<div style="margin-top: 6px;">Compared with the data already on this computer, no documents or '
                     'countries were added or removed.</div>')
        sections.append(f'<div style="margin-top: 10px;"><strong>Documents and data</strong><br>{" ".join(parts)}{body}</div>')
    if not sections:
        return ''
    return (
        '<div style="margin: 0 0 16px 0; padding: 14px 16px; border-radius: 10px; '
        'background: #e8f5e9; border-left: 5px solid #43a047; color: #1b3a1d;">'
        '<div style="font-weight: 700; margin-bottom: 2px;">NSDDD has been updated</div>'
        '<div style="font-size: 14px; line-height: 1.5;">'
        + ''.join(sections) +
        '<div style="margin-top: 10px; font-size: 13px; opacity: 0.85;">Updates are installed automatically each time you start NSDDD while you are online.</div>'
        '</div></div>'
    )


def build_update_banner_html() -> str:
    global NOTEBOOK_UPDATE_INFO

    _done = _updated_notice_html()
    if _done:
        return _done

    if NOTEBOOK_UPDATE_INFO is None:
        NOTEBOOK_UPDATE_INFO = check_notebook_update()

    if not NOTEBOOK_UPDATE_INFO.get('update_available'):
        return ''

    local_version = escape(str(NOTEBOOK_UPDATE_INFO.get('local') or 'unknown'))
    remote_version = escape(str(NOTEBOOK_UPDATE_INFO.get('remote') or 'unknown'))
    return (
        '<div style="margin: 0 0 16px 0; padding: 14px 16px; border-radius: 10px; '
        'background: #fff4db; border-left: 5px solid #f59e0b; color: #5b3b00;">'
        '<div style="font-weight: 700; margin-bottom: 4px;">Update available</div>'
        '<div style="font-size: 14px; line-height: 1.5;">'
        f'You are running NSDDD notebook version <code>{local_version}</code>, and GitHub has '
        f'<code>{remote_version}</code>. '
        'Relaunch via <code>Launch.command</code>, <code>Launch.vbs</code>, or '
        '<code>python launch.py</code> to update safely.'
        '</div></div>'
    )


# Metadata dictionary to store all loaded metadata
METADATA = {}

# Membership over time (2026-10-04). The old loader kept every row whose 'Year left' cell was empty, and looked for that
# column by one exact spelling. EU.csv ('year left') and AU.csv ('Year left/suspended') never matched, so the whole file
# counted: the UK as an EU member after 2020, Greenland as an EU member, Georgia and Ukraine in the CIS, Azerbaijan and
# Georgia in the CSTO, SCO dialogue partners and observers, OSCE partners and ASEAN observers as members.
# _library/membership.py reads each file as a history (joined, left, observer, suspended...) and gives intervals.
from _library import membership as _MB

_ORG_FILES = {
    'NATO': 'NATO.csv',
    'EU': 'EU.csv',
    'ASEAN': 'ASEAN.csv',
    'African Union': 'AU.csv',
    'BRICS': 'BRICS.csv',
    'Commonwealth': 'Commonwealth.csv',
    'G7': 'G7.csv',
    'G20': 'G20.csv',
    'G77': 'G77.csv',
    'OECD': 'OECD.csv',
    'Arab League': 'Arab_League.csv',
    'GCC': 'GCC_Gulf_Cooperation_Council.csv',
    'OAS': 'OAS.csv',
    'OSCE': 'OSCE.csv',
    'SCO': 'SCO.csv',
    'CSTO': 'CSTO.csv',
    'CIS': 'CIS.csv',
    'SIDS': 'SIDS.csv',
    'LDCs': 'LDCs_Least_Developed_Countries.csv',
    'NAM': 'NAM_Non_Aligned_Movement.csv',
    'P5': 'P5_UN_Security_Council.csv',
    'Francophonie': 'Francophonie.csv',
    'United Nations': 'UN.csv',
    'ANZUS': 'ANZUS.csv',
    'AUKUS': 'AUKUS.csv',
    'UFM': 'UFM.csv',
    'DCAF': 'DCAF.csv',
}


def load_organizational_history(filename: str) -> Dict[str, list]:
    """{iso: [(first_year, year_left_or_None), ...]} for one membership file."""
    filepath = BASE_PATH / filename
    if not filepath.exists():
        print(f'⚠️  {filename} not found')
        return {}
    try:
        return _MB.load_history(filepath)
    except Exception as e:
        print(f'⚠️  Error loading {filename}: {e}')
        return {}


def load_organizational_membership(filename: str) -> Set[str]:
    """ISO Alpha-3 codes of the CURRENT members in one membership file."""
    return _MB.current_members(load_organizational_history(filename))


print('\n🔄 Loading organizational metadata...')
membership_history = {org: load_organizational_history(f) for org, f in _ORG_FILES.items()}
organizations = {org: _MB.current_members(h) for org, h in membership_history.items()}

METADATA['organizations'] = organizations
METADATA['membership_history'] = membership_history

for org, members in organizations.items():
    if members:
        print(f'  ✓ {org}: {len(members)} countries with documents in the dataset')

# Load UN regional codes
print('\n🔄 Loading UN regional classifications...')
try:
    iso_df = pd.read_csv(BASE_PATH / 'ISO-3166-Countries-with-Regional-Codes.csv')

    # Create region mappings
    METADATA['regions'] = dict(zip(iso_df['alpha-3'], iso_df['region']))
    METADATA['subregions'] = dict(zip(iso_df['alpha-3'], iso_df['sub-region']))
    METADATA['intermediate_regions'] = dict(zip(iso_df['alpha-3'], iso_df['intermediate-region']))

    # Get unique regions and subregions
    unique_regions = sorted(iso_df['region'].dropna().unique())
    unique_subregions = sorted(iso_df['sub-region'].dropna().unique())

    print(f'  ✓ Regions: {len(unique_regions)} ({", ".join(unique_regions)})')
    print(f'  ✓ Subregions: {len(unique_subregions)}')

    METADATA['unique_regions'] = unique_regions
    METADATA['unique_subregions'] = unique_subregions

except Exception as e:
    print(f'⚠️  Error loading regional data: {e}')

# Load Freedom House data
print('\n🔄 Loading Freedom House democracy ratings...')
try:
    fh_df = pd.read_csv(BASE_PATH / 'Freedom_house.csv')

    # Freedom House uses columns for status, with years as values
    # Find the ISO column
    iso_col = [c for c in fh_df.columns if 'iso' in c.lower()][0]

    # Reshape to get most recent status
    freedom_status = {}
    for _, row in fh_df.iterrows():
        country_code = row[iso_col]
        # Check which status column has a value
        if pd.notna(row.get('Free')):
            freedom_status[country_code] = 'Free'
        elif pd.notna(row.get('Partly Free')):
            freedom_status[country_code] = 'Partly Free'
        elif pd.notna(row.get('Not Free')):
            freedom_status[country_code] = 'Not Free'

    METADATA['freedom_house'] = freedom_status

    # Count by status
    status_counts = Counter(freedom_status.values())
    for status, count in status_counts.items():
        print(f'  ✓ {status}: {count} countries')

except Exception as e:
    print(f'⚠️  Error loading Freedom House data: {e}')

# Load ODA data
print('\n🔄 Loading ODA recipient status...')
try:
    oda_df = pd.read_csv(BASE_PATH / 'ODA.csv')
    # Get latest status for each country
    oda_latest = oda_df.sort_values('Year', ascending=False).groupby('ISO_Code').first().reset_index()
    oda_mapping = dict(zip(oda_latest['ISO_Code'], oda_latest['Status']))
    METADATA['oda_mapping'] = oda_mapping
    print(f'  ✓ ODA data loaded ({len(oda_latest)} countries with status)')

    # Count countries by status
    oda_counts = oda_latest['Status'].value_counts()
    for status, count in oda_counts.items():
        print(f'  ✓ {status}: {count} countries')

except Exception as e:
    print(f'⚠️  Error loading ODA data: {e}')
    oda_mapping = {}
    METADATA['oda_mapping'] = oda_mapping

# Load World Bank income classifications
print('\n🔄 Loading World Bank income classifications...')
try:
    # OGHIST.csv has complex header structure - skip first 11 rows
    wb_df = pd.read_csv(BASE_PATH / 'OGHIST.csv', skiprows=11, header=None)

    # Columns: 0=ISO3, 1=Country, 2+=Classifications by year
    wb_df.columns = ['ISO3', 'Country'] + [f'Year_{i}' for i in range(len(wb_df.columns) - 2)]

    # Get most recent non-empty classification for each country
    income_groups = {}
    for _, row in wb_df.iterrows():
        iso_code = row['ISO3']
        # Get last non-empty classification
        classifications = [row[col] for col in wb_df.columns[2:] if pd.notna(row[col]) and row[col] not in ['..', '']]
        if classifications:
            last_class = classifications[-1]
            # Map abbreviations to full names
            class_map = {'L': 'Low', 'LM': 'Lower-middle', 'UM': 'Upper-middle', 'H': 'High'}
            income_groups[iso_code] = class_map.get(last_class, last_class)

    METADATA['income_groups'] = income_groups

    # Count by income group
    income_counts = Counter(income_groups.values())
    for income, count in sorted(income_counts.items()):
        print(f'  ✓ {income}: {count} countries')

except Exception as e:
    print(f'⚠️  Error loading World Bank data: {e}')

# Load document metadata
print('\n🔄 Loading document metadata...')
try:
    doc_df = pd.read_csv(BASE_PATH / 'document_metadata.csv')
    # Drop incomplete rows and normalise File prefix to integer strings (e.g. '416' not '416.0')
    doc_df = doc_df.dropna(subset=['File prefix'])
    doc_df['File prefix'] = doc_df['File prefix'].astype(int).astype(str)
    doc_df['Year'] = pd.to_numeric(doc_df['Year'], errors='coerce').astype('Int64')
    METADATA['documents'] = doc_df
    print(f'  ✓ Loaded {len(doc_df)} documents')
    print(f'  ✓ Countries: {doc_df["ISO Alpha-3 Code"].nunique()}')
    print(f'  ✓ Year range: {doc_df["Year"].min()}–{doc_df["Year"].max()}')
    print(f'  ✓ Document types: {", ".join(doc_df["NSS/WP/DD"].dropna().unique())}')

except Exception as e:
    print(f'⚠️  Error loading document metadata: {e}')


# Load Small States classifications
print('\n🔄 Loading Small States classifications...')
try:
    small_states_df = pd.read_csv(BASE_PATH / 'Small_States.csv')

    # Get ISO code column
    iso_col = 'ISO-Alpha3 Code'

    # Create sets for each classification
    small_states = set(small_states_df[small_states_df['Small_States'] == 'Yes'][iso_col].dropna().str.strip())
    ssf_members = set(small_states_df[small_states_df['SSF_Member'] == 'Yes'][iso_col].dropna().str.strip())
    un_sids_members = set(small_states_df[small_states_df['UN_SIDS_Member'] == 'Yes'][iso_col].dropna().str.strip())

    METADATA['small_states'] = small_states
    METADATA['ssf_members'] = ssf_members
    METADATA['un_sids_members'] = un_sids_members

    print(f'  ✓ Small States: {len(small_states)} countries')
    print(f'  ✓ Small States Forum (SSF): {len(ssf_members)} countries')
    print(f'  ✓ UN SIDS: {len(un_sids_members)} countries')

except Exception as e:
    print(f'⚠️  Error loading Small States data: {e}')

# Load Huntington's civilizational classifications
print('\n🔄 Loading Huntington civilizational classifications...')
try:
    hunt_df = pd.read_csv(BASE_PATH / 'Huntington_classification.csv')

    # Create mapping: ISO code -> classification
    huntington_map = dict(zip(hunt_df['ISO-Alpha3 Code'].str.strip(),
                             hunt_df['Huntingdons_class'].str.strip()))

    METADATA['huntington'] = huntington_map

    # Get unique classifications
    unique_civs = sorted(hunt_df['Huntingdons_class'].dropna().unique())
    METADATA['huntington_civilizations'] = unique_civs

    civ_counts = Counter(huntington_map.values())
    print(f'  ✓ Loaded {len(huntington_map)} country classifications')
    print(f'  ✓ Civilizations: {", ".join(unique_civs[:5])}...')

except Exception as e:
    print(f'⚠️  Error loading Huntington data: {e}')

print('\n✅ All metadata loaded successfully!')
print(f'\n📊 Total metadata categories: {len(METADATA)}')



# Load dataset metadata for normalisation context
COUNTRY_NORM_STATS: dict = {}
DOC_NORM_STATS: dict = {}  # per-document word_count keyed by doc_id string
try:
    _csv_path = BASE_PATH / 'document_metadata.csv'
    if not _csv_path.exists():
        _csv_path = ROOT_PATH / 'document_metadata_4.0.csv'
    if _csv_path.exists():
        _meta_df = pd.read_csv(_csv_path)
        for _country, _grp in _meta_df.groupby('Country'):
            COUNTRY_NORM_STATS[_country] = {
                'doc_count': len(_grp),
                'year_min': int(_grp['Year'].min()),
                'year_max': int(_grp['Year'].max()),
                'avg_words': int(round(_grp['word_count'].mean())),
            }
        # Per-document word counts for results-based normalisation
        if 'File prefix' in _meta_df.columns:
            for _, _dr in _meta_df.iterrows():
                _did = str(int(_dr['File prefix'])) if pd.notna(_dr['File prefix']) else None
                if _did:
                    _wc = _dr.get('word_count', None)
                    DOC_NORM_STATS[_did] = {
                        'word_count': int(_wc) if pd.notna(_wc) else 0,
                    }
except Exception:
    pass


In [ ]:
%%capture
@dataclass
class MetadataFilters:
    """Comprehensive metadata filtering options."""

    # Geographic filters
    regions: List[str] = field(default_factory=list)
    subregions: List[str] = field(default_factory=list)
    countries: List[str] = field(default_factory=list)

    # Organizational filters
    organizations: List[str] = field(default_factory=list)

    # Economic/political filters
    income_groups: List[str] = field(default_factory=list)
    freedom_house_status: List[str] = field(default_factory=list)
    oda_status: List[str] = field(default_factory=list)

    # Small States and civilizational filters
    small_states_groups: List[str] = field(default_factory=list)  # 'Small States', 'SSF', 'UN SIDS'
    huntington_civilizations: List[str] = field(default_factory=list)

    # Document filters
    doc_types: List[str] = field(default_factory=list)
    year_min: Optional[int] = None
    year_max: Optional[int] = None
    most_recent_only: bool = False
    membership_at_publication: bool = True    # organisations: members in the year each document was published (False = members today)

    def to_dict(self) -> Dict:
        """Convert to dictionary for display."""
        return {k: v for k, v in asdict(self).items() if v not in [[], None, False]}

    def get_summary(self) -> str:
        """Get human-readable summary of active filters."""
        active = []

        if self.regions:
            active.append(f"Regions: {', '.join(self.regions)}")
        if self.subregions:
            active.append(f"Subregions: {', '.join(self.subregions[:3])}{'...' if len(self.subregions) > 3 else ''}")
        if self.countries:
            active.append(f"Countries: {', '.join(self.countries[:5])}{'...' if len(self.countries) > 5 else ''}")
        if self.organizations:
            active.append(f"Organizations: {', '.join(self.organizations)}"
                          + (' (members when each document was published)' if self.membership_at_publication else ' (current members)'))
        if self.income_groups:
            active.append(f"Income: {', '.join(self.income_groups)}")
        if self.freedom_house_status:
            active.append(f"Democracy: {', '.join(self.freedom_house_status)}")
        if self.oda_status:
            active.append(f"ODA: {', '.join(self.oda_status)}")
        if self.small_states_groups:
            active.append(f"Small States: {', '.join(self.small_states_groups)}")
        if self.huntington_civilizations:
            active.append(f"Civilizations: {', '.join(self.huntington_civilizations[:3])}{'...' if len(self.huntington_civilizations) > 3 else ''}")
        if self.doc_types:
            active.append(f"Doc types: {', '.join(self.doc_types)}")
        if self.year_min is not None or self.year_max is not None:
            yr_range = f"{self.year_min if self.year_min is not None else 'start'}–{self.year_max if self.year_max is not None else 'end'}"
            active.append(f"Years: {yr_range}")
        if self.most_recent_only:
            active.append("Most recent docs only")

        return ' | '.join(active) if active else 'No filters applied'

@dataclass
class SearchConfig:
    """Search configuration."""
    topics: List[str]
    search_threshold: float = 0.72
    cluster_threshold: float = 0.78
    export_format: str = 'xlsx'       # 'xlsx' = Excel workbook with clickable links; 'csv' = CSV with plain web addresses
    export_prefix: str = 'metadata_search_'
    export_enabled: bool = True
    keyword_search: bool = False
    fuzzy_matching: bool = False
    fuzzy_threshold: int = 80
    smart_boundaries: bool = True  # Allow hyphenated variants
    context_window: int = 0
    max_results_per_topic: Optional[int] = None
    count_passages_once: bool = True   # an original-language match and the English match of the same passage count as one result

    # Boolean search configuration
    boolean_mode: bool = False  # Enable Boolean operators (AND, OR, NOT)
    boolean_scoring: str = 'match_count'  # Scoring strategy: 'match_count', 'avg_score', or 'max_score'

    # Metadata filters
    filters: MetadataFilters = field(default_factory=MetadataFilters)

@dataclass
class SearchResult:
    """Individual search result."""
    segment_id: str
    text: str
    score: float
    document_id: str
    document_name: str
    country: str
    country_iso: str  # ISO Alpha-3 code for metadata lookups
    year: int
    doc_type: str
    context_before: str = ''
    context_after: str = ''
    cluster_id: Optional[str] = None
    cluster_name: str = ''  # 'Cluster 3', or an AI heading once the user has added them
    cluster_example: str = ''  # the cluster's most typical sentence (up to 30 words)

    # Metadata enrichment
    region: str = ''
    subregion: str = ''
    income_group: str = ''
    freedom_status: str = ''

print('✅ Data structures defined')

In [ ]:
%%capture
# ================================
# Section 4: Model + Search Utilities (FULL CELL)
# ================================

# Global model storage
MODEL_DICT = None
ENCODER = None
MODEL_LOAD_ERROR = None
ENCODER_LOAD_ERROR = None

def initialize_model(load_encoder=False):
    """Initialize semantic search model."""
    global MODEL_DICT, ENCODER, MODEL_LOAD_ERROR, ENCODER_LOAD_ERROR

    if MODEL_DICT is not None:
        if load_encoder:
            initialize_encoder()
        return MODEL_DICT, ENCODER

    print('🔄 Loading model files...')
    MODEL_LOAD_ERROR = None
    ENCODER_LOAD_ERROR = None

    try:
        with open(MODEL_PATH / 'documents_dict.json', 'r', encoding='utf-8') as f:
            documents_dict = json.load(f)

        with open(MODEL_PATH / 'countries_dict.json', 'r', encoding='utf-8') as f:
            countries_dict = json.load(f)

        # Qwen3 app: segment texts are read from SQLite on demand (not held in memory)
        if (MODEL_PATH / 'segments.sqlite').exists():
            from _library.segment_store import SegmentStore
            segments_dict = SegmentStore(MODEL_PATH / 'segments.sqlite')
        else:
            with open(MODEL_PATH / 'segments_dict.json', 'r', encoding='utf-8') as f:
                segments_dict = json.load(f)

        with open(MODEL_PATH / 'encoded_segments.json', 'r', encoding='utf-8') as f:
            encoded_segments = json.load(f)

        alignment_dict = {}
        _align_path = MODEL_PATH / 'alignment_dict.json'
        if _align_path.exists():
            with open(_align_path, 'r', encoding='utf-8') as f:
                alignment_dict = json.load(f)

        alignment_anchor = {}
        _anchor_path = MODEL_PATH / 'alignment_anchor.json'
        if _anchor_path.exists():
            with open(_anchor_path, 'r', encoding='utf-8') as f:
                alignment_anchor = json.load(f)

        print('  ✓ Loading segment encodings...')
        _meta_path = MODEL_PATH / 'segment_encodings_metadata.json'
        _memmap_path = MODEL_PATH / 'segment_encodings.f32.memmap'
        if _meta_path.exists():                     # Qwen3 app: float16 index named in the metadata
            _mf = json.loads(_meta_path.read_text(encoding='utf-8')).get('memmap_file')
            if _mf:
                _memmap_path = MODEL_PATH / _mf
        _npy_path = MODEL_PATH / 'segment_encodings.npy'

        if _meta_path.exists() and _memmap_path.exists():
            import json as _json
            with open(_meta_path, 'r', encoding='utf-8') as _f:
                _meta = _json.load(_f)

            _shape = _meta.get('memmap_shape')
            if not _shape:
                _n_rows = _meta.get('n_rows')
                _dim = _meta.get('embedding_dim')
                if _n_rows is None or _dim is None:
                    raise KeyError('Metadata missing shape keys: memmap_shape or n_rows+embedding_dim')
                _shape = (_n_rows, _dim)
            _shape = tuple(_shape)

            _dtype = _meta.get('memmap_dtype') or _meta.get('dtype') or 'float32'
            segment_encodings = np.memmap(
                str(_memmap_path), dtype=_dtype, mode='r', shape=_shape
            )
            print(f'    • Using {_dtype} memmap embeddings {_shape}')
        elif _npy_path.exists():
            segment_encodings = np.load(str(_npy_path), mmap_mode='r')
            print('    • Using .npy embeddings (memory-mapped)')
        else:
            print('    • Falling back to JSON embeddings (slower)')
            with open(MODEL_PATH / 'segment_encodings.json', 'r', encoding='utf-8') as f:
                segment_encodings = np.array(json.load(f), dtype=np.float32)

        _short = MODEL_PATH / 'short_mask.npy'
        _enc_meta = json.loads(_meta_path.read_text(encoding='utf-8')) if _meta_path.exists() else {}
        _tor, _tl = MODEL_PATH / 'topic_of_row.npy', MODEL_PATH / 'topic_labels.json'
        MODEL_DICT = {
            'model_dir': str(MODEL_PATH),
            # precomputed corpus topics (build_topic_labels_qwen3.py): headings for the result clusters
            'topic_of_row': np.load(_tor) if _tor.exists() else None,
            'topic_labels': ({k: v['label'] for k, v in json.loads(_tl.read_text(encoding='utf-8')).items()}
                             if _tl.exists() else {}),
            'short_mask': np.load(_short) if _short.exists() else None,          # segments of <= 6 words
            'short_penalty': float(_enc_meta.get('short_penalty', 0.05)),       # ranked lower (bake-off '+hp')
            'encoder_meta': _enc_meta,
            'documents_dict': documents_dict,
            'countries_dict': countries_dict,
            'segments_dict': segments_dict,
            'encoded_segments': encoded_segments,
            'alignment_dict': alignment_dict,
            'alignment_anchor': alignment_anchor,
            'segment_encodings': segment_encodings
        }

        print(f'\n✅ Model loaded successfully!')
        print(f'   📄 Documents: {len(documents_dict):,}')
        print(f'   🌍 Countries: {len(countries_dict):,}')
        print(f'   📝 Segments: {len(segments_dict):,}')
        print(f'   🔢 Encoded segments: {len(encoded_segments):,}')

        if load_encoder:
            initialize_encoder()
        return MODEL_DICT, ENCODER

    except Exception as e:
        print(f'❌ Error loading model: {e}')
        import traceback
        MODEL_LOAD_ERROR = traceback.format_exc()
        print(MODEL_LOAD_ERROR)
        return None, None


def initialize_encoder():
    global ENCODER, ENCODER_LOAD_ERROR
    if ENCODER is not None:
        return ENCODER
    if MODEL_DICT is None:
        initialize_model()
    if MODEL_DICT is None:
        return None
    # Required only for semantic searches.
    try:
        from sentence_transformers import SentenceTransformer
        import logging
        import warnings

        # Suppress model loading warnings
        import os
        os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress TensorFlow warnings
        os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # Suppress tokenizer warnings

        warnings.filterwarnings('ignore')
        logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
        logging.getLogger('transformers').setLevel(logging.ERROR)

        # Suppress transformers model loading verbosity
        from transformers import logging as transformers_logging
        transformers_logging.set_verbosity_error()

        _em = MODEL_DICT.get('encoder_meta') or {}
        _local = MODEL_PATH / _em.get('local_model_dir', 'qwen3-embedding-0.6b')
        if not _local.is_dir():
            raise FileNotFoundError(f'Local semantic encoder is missing: {_local}. Rerun setup.')
        import torch
        torch.set_num_threads(int(os.environ.get('NSDDD_THREADS', os.cpu_count() or 2)))
        # bfloat16 weights: half the memory of float32 (~1.1 GB), query vectors identical (cosine 0.9998)
        _st = SentenceTransformer(str(_local),
                                  device='cpu', model_kwargs={'torch_dtype': torch.bfloat16})

        class _QueryEncoder:
            """Qwen3-Embedding-0.6B query side: the model's 'query' prompt, cut to the index's dimensions
            (Matryoshka) and L2-normalised, so encode(text) matches the stored segment vectors."""
            def __init__(self, model, dims, prompt):
                self.model, self.dims, self.prompt = model, dims, prompt

            def encode(self, texts, **kw):
                one = isinstance(texts, str)
                kw.pop('normalize_embeddings', None)
                v = np.asarray(self.model.encode([texts] if one else list(texts), prompt_name=self.prompt,
                                                 normalize_embeddings=True, **kw), dtype=np.float32)[:, :self.dims]
                v = v / np.clip(np.linalg.norm(v, axis=1, keepdims=True), 1e-10, None)
                return v[0] if one else v

        ENCODER = _QueryEncoder(_st, int(_em.get('dim', 256)), _em.get('query_prompt_name', 'query'))
        print('  ✓ Encoder initialized')
    except Exception as e:
        import traceback
        ENCODER_LOAD_ERROR = traceback.format_exc()
        print('  ⚠️  Could not load encoder (keyword search will still work)')
        print(ENCODER_LOAD_ERROR)
        ENCODER = None

    return ENCODER


def apply_metadata_filters(doc_ids: List[str], filters: 'MetadataFilters') -> List[str]:
    """
    Apply metadata filters to a list of document IDs (File prefix strings).

    Behaviour:
    - Multiple selections within a category are OR (union).
    - Selections across categories are AND (intersection).

    Returns:
        List[str] of filtered document IDs (File prefix values as strings).
    """
    if filters is None or not filters.to_dict():
        return [str(d) for d in doc_ids]

    if 'documents' not in METADATA:
        raise KeyError("METADATA['documents'] is missing. Load document_metadata.csv first.")

    doc_df = METADATA['documents']
    filtered_docs = doc_df[doc_df['File prefix'].astype(str).isin([str(d) for d in doc_ids])]

    valid_countries = None

    def constrain(codes):
        nonlocal valid_countries
        codes = set(codes)
        valid_countries = codes if valid_countries is None else valid_countries & codes

    # Direct country selection (ISO3)
    if getattr(filters, 'countries', None):
        constrain(filters.countries)

    # OR within a category, AND across independently selected categories.
    for field, mapping in [('regions', 'regions'), ('subregions', 'subregions')]:
        selected = getattr(filters, field, None)
        if selected:
            constrain(code for code, value in METADATA.get(mapping, {}).items() if value in selected)

    # Organizations: current members here; members in the document's own year are applied per document further down
    if getattr(filters, 'organizations', None) and not getattr(filters, 'membership_at_publication', False):
        org_countries: Set[str] = set()
        for org in filters.organizations:
            org_countries.update(METADATA.get('organizations', {}).get(org, set()))
        constrain(org_countries)

    # Income groups
    if getattr(filters, 'income_groups', None):
        income_countries = {
            code for code, group in METADATA.get('income_groups', {}).items()
            if group in filters.income_groups
        }
        constrain(income_countries)

    # Freedom House
    if getattr(filters, 'freedom_house_status', None):
        fh_countries = {
            code for code, status in METADATA.get('freedom_house', {}).items()
            if status in filters.freedom_house_status
        }
        constrain(fh_countries)

    # ODA status
    if getattr(filters, 'oda_status', None):
        oda_countries = {
            code for code, status in METADATA.get('oda_mapping', {}).items()
            if status in filters.oda_status
        }
        constrain(oda_countries)

    # Small states groups
    if getattr(filters, 'small_states_groups', None):
        ss_countries: Set[str] = set()
        for group in filters.small_states_groups:
            if group == 'Small States':
                ss_countries.update(METADATA.get('small_states', set()))
            elif group == 'SSF':
                ss_countries.update(METADATA.get('ssf_members', set()))
            elif group == 'UN SIDS':
                ss_countries.update(METADATA.get('un_sids_members', set()))
        constrain(ss_countries)

    # Huntington civilizations
    if getattr(filters, 'huntington_civilizations', None):
        hunt_countries = {
            code for code, civ in METADATA.get('huntington', {}).items()
            if civ in filters.huntington_civilizations
        }
        constrain(hunt_countries)

    # Apply country filter if any countries are constrained
    if valid_countries is not None:
        filtered_docs = filtered_docs[filtered_docs['ISO Alpha-3 Code'].isin(valid_countries)]

    # Organizations, counted in the year each document was published (accession year inclusive, year left exclusive)
    if getattr(filters, 'organizations', None) and getattr(filters, 'membership_at_publication', False):
        _hist = METADATA.get('membership_history', {})
        _orgs = [o for o in filters.organizations if o in _hist]
        filtered_docs = filtered_docs.loc[[
            any(_MB.is_member(_hist[o], str(iso), int(yr) if pd.notna(yr) else 0) for o in _orgs)
            for iso, yr in zip(filtered_docs['ISO Alpha-3 Code'], filtered_docs['Year'])
        ]]

    # Doc type filter
    if getattr(filters, 'doc_types', None):
        filtered_docs = filtered_docs[filtered_docs['NSS/WP/DD'].isin(filters.doc_types)]

    # Year range filter
    if getattr(filters, 'year_min', None) is not None:
        filtered_docs = filtered_docs[filtered_docs['Year'] >= filters.year_min]
    if getattr(filters, 'year_max', None) is not None:
        filtered_docs = filtered_docs[filtered_docs['Year'] <= filters.year_max]

    # Most recent per country
    if getattr(filters, 'most_recent_only', False):
        filtered_docs = (
            filtered_docs.sort_values('Year')
            .groupby('ISO Alpha-3 Code')
            .last()
            .reset_index()
        )

    return filtered_docs['File prefix'].astype(str).tolist()


def _clean_metadata_value(value):
    """Normalise metadata values so missing data never reaches the UI as NaN."""
    if value is None:
        return ''
    try:
        if pd.isna(value):
            return ''
    except Exception:
        pass
    return str(value).strip()


_REGION_FALLBACK = {'TWN': ('Asia', 'Eastern Asia'), 'SCG': ('Europe', 'Southern Europe')}   # UN geoscheme, as in the reference file


def enrich_result_with_metadata(result: 'SearchResult') -> 'SearchResult':
    """Add region/subregion/income/freedom metadata to a SearchResult (in place)."""
    country_code = getattr(result, 'country_iso', '') or ''
    result.region = _clean_metadata_value(METADATA.get('regions', {}).get(country_code, ''))
    result.subregion = _clean_metadata_value(METADATA.get('subregions', {}).get(country_code, ''))
    if not result.region and country_code in _REGION_FALLBACK:     # Taiwan and Serbia and Montenegro are not in the ISO regional file
        result.region, result.subregion = _REGION_FALLBACK[country_code]
    result.income_group = _clean_metadata_value(METADATA.get('income_groups', {}).get(country_code, ''))
    result.freedom_status = _clean_metadata_value(METADATA.get('freedom_house', {}).get(country_code, ''))
    return result


def get_segment_text_safe(segment_data: Dict) -> Optional[str]:
    """Safely extract text from segment dict with fallback keys."""
    if not segment_data:
        return None
    for key in ['cleaned', 'text', 'segment', 'content', 'sentence', 'raw']:
        val = segment_data.get(key)
        if val:
            return str(val).strip()
    return None


def create_keyword_pattern(keyword: str, smart_boundaries: bool = True) -> re.Pattern:
    """
    Create a compiled regex pattern for keyword/phrase matching with wildcard support.

    - Robust tokenisation: split ORIGINAL keyword on whitespace, then escape tokens.
    - Wildcards:
        * -> matches any characters (non-greedy)
        ? -> matches any single character
    - Smart phrase matching (optional): allows unicode dashes and separators between tokens:
      -, /, – (U+2013), — (U+2014), ‑ (U+2011), − (U+2212), soft hyphen (U+00AD), plus whitespace.
    - Safer boundaries: uses letter/digit boundaries rather than \\b.

    Returns:
        Compiled pattern (IGNORECASE | MULTILINE).
    """
    MATCH_NOTHING = re.compile(r'(?!)')

    if keyword is None:
        return MATCH_NOTHING

    keyword = str(keyword).strip()
    if not keyword:
        return MATCH_NOTHING

    raw_tokens = re.findall(r"\S+", keyword)
    if not raw_tokens:
        return MATCH_NOTHING

    def token_to_regex(token: str) -> str:
        escaped = re.escape(token)
        escaped = escaped.replace(r"\*", r".*?")  # non-greedy wildcard
        escaped = escaped.replace(r"\?", r".")    # single char wildcard
        return escaped

    tokens = [token_to_regex(t) for t in raw_tokens]

    sep = r"[\s\-/–—‑−\u00AD]*"
    start_boundary = r"(?<![A-Za-z0-9])"
    end_boundary = r"(?![A-Za-z0-9])"

    if len(tokens) == 1:
        pattern_str = start_boundary + tokens[0] + end_boundary
    else:
        if smart_boundaries:
            body = sep.join(tokens)
        else:
            body = r"\s+".join(tokens)
        pattern_str = start_boundary + body + end_boundary

    return re.compile(pattern_str, re.IGNORECASE | re.MULTILINE)


def get_context_window(segment_id: str, model_dict: Dict, window_size: int = 1) -> Tuple[str, str]:
    """
    Get surrounding segment text (before, after) for a segment_id like "DOCID/12".
    """
    if window_size <= 0:
        return '', ''

    if not model_dict or 'segments_dict' not in model_dict:
        return '', ''

    try:
        doc_id, segment_num = str(segment_id).rsplit('/', 1)      # '545/662' or an original-language id '642/o/44'
        segment_num = int(segment_num)
    except Exception:
        return '', ''

    segs = model_dict['segments_dict']
    before_parts = []
    after_parts = []

    for i in range(max(1, segment_num - window_size), segment_num):
        sid = f"{doc_id}/{i}"
        txt = get_segment_text_safe(segs.get(sid, {}))
        if txt:
            before_parts.append(txt)

    for i in range(segment_num + 1, segment_num + window_size + 1):
        sid = f"{doc_id}/{i}"
        txt = get_segment_text_safe(segs.get(sid, {}))
        if txt:
            after_parts.append(txt)

    return ' '.join(before_parts), ' '.join(after_parts)


print('✅ Section 4 loaded: initialize_model, apply_metadata_filters, keyword helpers, context helpers')


In [ ]:
%%capture
def apply_cluster_names(results: List[SearchResult], cluster_names: Dict[str, str]):
    """
    Apply cluster names to SearchResult objects in place.
    
    Args:
        results: List of SearchResult objects
        cluster_names: Dictionary mapping cluster_id -> cluster_name
    """
    for result in results:
        cluster_id = result.cluster_id
        if cluster_id is not None and cluster_id in cluster_names:
            result.cluster_name = cluster_names[cluster_id]
        elif cluster_id == 'singletons':
            result.cluster_name = cluster_names.get('singletons', 'Unclustered')
        else:
            result.cluster_name = 'Cluster ' + str(cluster_id).replace('cluster_', '')

def cluster_search_results_internal(
    results: List[SearchResult],
    model_dict: Dict,
    cluster_threshold: float = 0.78,
    search_query: str = ''
) -> Dict:
    """
    Shared clustering function for both similarity and keyword searches.

    Takes the Qwen3 embeddings of the result segments and groups them by Ward clustering. Clusters are numbered
    1, 2, 3... from the largest down; they are not given keyword headings.

    Args:
        results: List of SearchResult objects with segment_id populated
        model_dict: Dictionary containing segment_encodings and encoded_segments
        cluster_threshold: Similarity threshold for clustering (default 0.78)

    Returns:
        Dictionary with keys:
            - cluster_dict: {cluster_id: [result_indices]} mapping
            - cluster_labels: array of cluster_id per result
            - num_clusters: count of clusters with 2+ members
            - num_singletons: count of unclustered results
            - cluster_names: {cluster_id: 'Cluster N'} (AI headings replace these only on request, see apply_ai_cluster_labels)
            - label_inputs: {cluster_id: [(sentence, document_id)]} what the optional AI heading model reads
    """
    from scipy.spatial.distance import pdist, squareform
    from scipy.sparse.csgraph import connected_components
    import numpy as np

    limit = int(os.environ.get('NSDDD_CLUSTER_LIMIT', '3000' if LOW_POWER else '6000'))
    if len(results) > limit:
        reason = f'Clustering skipped for {len(results):,} results (memory limit: {limit:,}); all results are retained.'
        print(reason)
        for result in results:
            result.cluster_id, result.cluster_name = 'singletons', 'Unclustered'
        return {'cluster_dict': {'singletons': list(range(len(results)))},
                'num_clusters': 0, 'num_singletons': len(results),
                'cluster_names': {'singletons': 'Unclustered'}, 'clustering_note': reason,
                'label_inputs': {}, 'cluster_examples': {}, 'label_source': 'numbers'}

    # Qwen3 app: read only the result rows from the index (the full index was copied into memory here before)
    segment_encodings = model_dict['segment_encodings']
    encoded_segments = model_dict['encoded_segments']
    segment_to_idx = model_dict.get('_row_of')
    if segment_to_idx is None:                      # built once, then cached
        segment_to_idx = model_dict['_row_of'] = {seg_id: idx for idx, seg_id in enumerate(encoded_segments)}

    rows, valid_indices = [], []
    for i, result in enumerate(results):
        enc_idx = segment_to_idx.get(result.segment_id)
        if enc_idx is not None:
            rows.append(enc_idx)
            valid_indices.append(i)

    if len(rows) == 0:
        raise ValueError('No valid embeddings found for results')

    result_embeddings = np.asarray(segment_encodings[np.array(rows)], dtype=np.float32)
    result_embeddings /= np.clip(np.linalg.norm(result_embeddings, axis=1, keepdims=True), 1e-10, None)

    # Qwen3 app: Ward clustering (2026-10-01). Linking every pair above a threshold chained almost all results into
    # one giant cluster (94% of 3,950 for 'climate change is a security threat'); Ward gives balanced, deterministic
    # clusters (largest 10%). The number of clusters grows with the results (about sqrt(n / 10): 20 for 4,000,
    # 2-3 for 60); the 'Cluster detail' slider scales it: +0.04 above 0.78 doubles it, -0.04 halves it.
    from scipy.cluster.hierarchy import linkage, fcluster
    _n = len(result_embeddings)
    if _n < 3:
        labels = np.arange(_n)
    else:
        _k = max(2.0, (_n / 10.0) ** 0.5) * 2.0 ** ((cluster_threshold - 0.78) / 0.04)
        _k = int(min(max(2, round(_k)), 60, _n - 1))
        labels = fcluster(linkage(result_embeddings, method='ward'), _k, criterion='maxclust') - 1
        _sizes = np.bincount(labels)                                          # number the clusters 1, 2, 3... largest first
        _rank = {int(l): r for r, l in enumerate(np.argsort(-_sizes, kind='stable'))}
        labels = np.array([_rank[int(l)] for l in labels])

    # Map back to original result indices
    full_labels = np.full(len(results), -1, dtype=int)
    for i, valid_idx in enumerate(valid_indices):
        full_labels[valid_idx] = labels[i]

    # Separate clusters (2+ members) from singletons
    cluster_dict = {}
    singleton_indices = []

    # Create mapping from label to original result indices
    label_to_indices = {}
    for i, valid_idx in enumerate(valid_indices):
        label = labels[i]
        if label not in label_to_indices:
            label_to_indices[label] = []
        label_to_indices[label].append(valid_idx)

    # Separate clusters (2+ members) from singletons
    for label, indices in label_to_indices.items():
        if len(indices) >= 2:
            cluster_id = f'cluster_{int(label) + 1}'
            cluster_dict[cluster_id] = indices
        else:
            singleton_indices.extend(indices)

    # Add any results without valid embeddings to singletons
    valid_set = set(valid_indices)
    for i in range(len(results)):
        if i not in valid_set:
            singleton_indices.append(i)

    if singleton_indices:
        cluster_dict['singletons'] = sorted(set(singleton_indices))

    # Build cluster info dictionary
    cluster_info = {
        'cluster_dict': cluster_dict,
        'cluster_labels': full_labels,
        'num_clusters': len([k for k in cluster_dict.keys() if k != 'singletons']),
        'num_singletons': len(singleton_indices)
    }

    # Assign cluster IDs to results FIRST (needed for name generation)
    for cluster_id, indices in cluster_dict.items():
        for idx in indices:
            results[idx].cluster_id = cluster_id if cluster_id != 'singletons' else 'singletons'

    # Cluster headings (2026-10-04): numbers only. The keyword headings were dropped after the Windows test: they read as
    # arbitrary lists ('Variable, nature, changed') and appeared beside AI headings with nothing to show which was which.
    # What identifies a cluster now is its number plus its most typical sentence ('example', shown on hover and in the
    # export). AI headings are optional and added afterwards by apply_ai_cluster_labels (the 'Add AI headings' button),
    # which reads `label_inputs`: each cluster's most central sentences, English preferred.
    import re
    cluster_examples, label_inputs = {}, {}

    def _short(t, n=30):
        t = re.sub(r'^[\W\d_]+', '', str(t)).strip()
        w = t.split()
        return ' '.join(w[:n]).rstrip(',;:') + ('…' if len(w) > n else '')

    try:
        _row = {vi: k for k, vi in enumerate(valid_indices)}
        for cid, idxs in cluster_dict.items():
            if cid == 'singletons':
                continue
            rows_ = [i for i in idxs if i in _row]
            if not rows_:
                continue
            V = result_embeddings[[_row[i] for i in rows_]]
            order = np.argsort(-(V @ V.mean(axis=0)))
            cand = [rows_[k] for k in order]
            en_only = [i for i in cand if '/o/' not in results[i].segment_id]
            en = en_only or cand
            pick = next((i for i in en if 5 <= len(results[i].text.split()) <= 30), en[0])
            cluster_examples[cid] = _short(results[pick].text)
            if en_only:                          # a cluster of original-language passages only keeps its number: the small model writes English headings from English text
                label_inputs[cid] = [(results[i].text, results[i].document_id) for i in en_only[:60]]
    except Exception:
        cluster_examples, label_inputs = {}, {}              # an example is a convenience: never let it break a search

    cluster_names = {cid: ('Unclustered' if cid == 'singletons' else 'Cluster ' + cid.split('_')[-1]) for cid in cluster_dict}
    cluster_info['cluster_examples'] = cluster_examples
    cluster_info['label_inputs'] = label_inputs
    cluster_info['label_source'] = 'numbers'
    for cid, idxs in cluster_dict.items():
        for i in idxs:
            results[i].cluster_example = cluster_examples.get(cid, '')
    cluster_info['cluster_names'] = cluster_names

    # Apply cluster names to results
    apply_cluster_names(results, cluster_names)

    return cluster_info


def apply_ai_cluster_labels(all_results, all_cluster_info, model_dict, progress=None):
    """Give a finished search's clusters AI-written headings (the 'Add AI headings' button).

    Clusters the small model cannot name acceptably keep their numbers; there is no keyword fallback. Names are written
    into cluster_info['cluster_names'] and onto the results, so a later display or export carries them.
    Returns (clusters named, clusters in total)."""
    from _library import cluster_labeller as _CLB
    named = total = 0
    for topic, results in all_results.items():
        ci = all_cluster_info.get(topic) or {}
        inputs = ci.get('label_inputs') or {}
        if not inputs:
            continue
        cd = ci['cluster_dict']
        total += len(inputs)

        def _cb(done, n, got, _base=named):
            if progress:
                progress(_base + got, total, done, n)
        ai = _CLB.label_clusters([{'id': cid, 'size': len(cd[cid]), 'sentences': inputs[cid]} for cid in inputs],
                                 model_dict.get('model_dir'), budget_s=60 if LOW_POWER else 180, progress=_cb)
        ci['cluster_names'].update(ai)
        ci['ai_named'] = sorted(ai)
        ci['label_source'] = 'ai' if len(ai) == len(inputs) else ('ai_partial' if ai else 'numbers')
        apply_cluster_names(results, ci['cluster_names'])
        named += len(ai)
    _CLB.unload()
    return named, total


In [ ]:
%%capture
def display_filter_summary(filters: MetadataFilters):
    """Display active filters in an attractive format."""
    summary = filters.get_summary()
    
    if summary == 'No filters applied':
        style = 'background-color: #f0f0f0; padding: 10px; border-radius: 5px; color: #666;'
        icon = '🌐'
    else:
        style = 'background-color: #e3f2fd; padding: 10px; border-radius: 5px; border-left: 4px solid #2196F3; color: #1565c0;'
        icon = '🔍'
    
    html = f"""
    <div style="{style}">
        <strong>{icon} Active Filters:</strong><br/>
        {summary}
    </div>
    """
    display(HTML(html))

def display_search_results(results: List[SearchResult], topic: str, filters: MetadataFilters, 
                          config: SearchConfig):
    """Display search results with metadata and filter information."""
    
    # Header
    display(HTML(f"""
    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                padding: 20px; border-radius: 10px; color: white; margin: 20px 0;">
        <h2 style="margin: 0; font-size: 24px;">🔍 Results for: "{topic}"</h2>
        <p style="margin: 10px 0 0 0; opacity: 0.9;">Found {len(results)} results in {len(set(r.document_id for r in results))} documents</p>
    </div>
    """))
    
    # Display active filters
    display_filter_summary(filters)
    
    # Search configuration
    search_type = 'Keyword Search' if config.keyword_search else 'Semantic Search'
    display(HTML(f"""
    <div style="background-color: #fff3e0; padding: 10px; margin: 10px 0; 
                border-radius: 5px; border-left: 4px solid #ff9800;">
        <strong>⚙️ Search Configuration:</strong> {search_type} | 
        Threshold: {config.search_threshold:.2f} | 
        Cluster: {config.cluster_threshold:.2f}
    </div>
    """))
    
    # Group results by document
    doc_groups = {}
    for r in results:
        if r.document_id not in doc_groups:
            doc_groups[r.document_id] = []
        doc_groups[r.document_id].append(r)
    
    # Display each document group
    for doc_id, doc_results in sorted(doc_groups.items()):
        first_result = doc_results[0]
        
        # Document header with metadata
        metadata_tags = []
        if first_result.region:
            metadata_tags.append(f"🌍 {first_result.region}")
        if first_result.subregion:
            metadata_tags.append(f"📍 {first_result.subregion}")
        if first_result.income_group:
            metadata_tags.append(f"💰 {first_result.income_group}")
        if first_result.freedom_status:
            metadata_tags.append(f"🗳️ {first_result.freedom_status}")
        
        metadata_str = ' | '.join(metadata_tags)
        
        display(HTML(f"""
        <div style="background-color: #f5f5f5; padding: 15px; margin: 15px 0; 
                    border-radius: 8px; border-left: 5px solid #4CAF50;">
            <h3 style="margin: 0 0 10px 0; color: #546e7a;">
                📄 {first_result.country} ({first_result.year})
            </h3>
            <p style="margin: 5px 0; color: #555; font-size: 14px;">
                <strong>{first_result.document_name}</strong>
            </p>
            <p style="margin: 5px 0; color: #777; font-size: 12px;">
                {metadata_str}
            </p>
            <p style="margin: 5px 0; color: #999; font-size: 12px;">
                📊 {len(doc_results)} results in this document
            </p>
        </div>
        """))
        
        # Display individual results
        for i, result in enumerate(doc_results[:5], 1):  # Show top 5 per document
            cluster_info = f" | Cluster: {result.cluster_id}" if result.cluster_id is not None else ""
            display(HTML(f"""
            <div style="background-color: white; padding: 12px; margin: 8px 0 8px 20px; 
                        border-radius: 5px; border-left: 3px solid #9C27B0;">
                <p style="margin: 0 0 5px 0; color: #888; font-size: 11px;">
                    Result {i} | Score: {result.score:.3f}{cluster_info}
                </p>
                <p style="margin: 0; color: #333; line-height: 1.6;">
                    {result.text[:500]}{'...' if len(result.text) > 500 else ''}
                </p>
            </div>
            """))
        
        if len(doc_results) > 5:
            display(HTML(f"""
            <p style="margin: 5px 0 5px 20px; color: #999; font-size: 12px; font-style: italic;">
                ... and {len(doc_results) - 5} more results
            </p>
            """))

def create_results_summary_chart(all_results: Dict[str, List[SearchResult]], filters: MetadataFilters):
    """Create summary visualization of search results."""
    if not all_results:
        return
    
    # Aggregate statistics
    topic_counts = {topic: len(results) for topic, results in all_results.items()}
    
    # Create bar chart
    fig, ax = plt.subplots(figsize=(12, 6))
    topics = list(topic_counts.keys())
    counts = list(topic_counts.values())
    
    bars = ax.barh(topics, counts, color='#667eea')
    ax.set_xlabel('Number of Results', fontsize=12)
    ax.set_title('Search Results Summary', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for bar in bars:
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2, 
               f' {int(width)}', va='center', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # Display filter summary
    display(Markdown(f"**Filters applied:** {filters.get_summary()}"))

print('✅ Display functions defined')

In [ ]:
%%capture
def generate_search_visualisations(all_results, all_cluster_info, save_dir=None, file_prefix=None, timestamp=None):
    """Generate SVG charts from search results.

    Returns an HTML string with charts embedded inline.
    If *save_dir* is provided, SVG files are also written to disk.

    The year chart includes two prevalence normalizations:
    - Yearly prevalence: matched docs / docs published that year
    - Current-doc prevalence: for each year, share of active countries whose
      latest-available document (<= year) contains at least one match
    """
    import io
    if not VISUALIZATION_AVAILABLE:
        return (
            '<div style="margin: 10px 0; padding: 10px 12px; '
            'background-color: #fff9e6; border-left: 3px solid #ffa726; color: #6b4f00;">'
            'Charts are unavailable because matplotlib is not installed in this environment.</div>'
        )

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from collections import Counter, defaultdict
    from pathlib import Path

    flat_results = [r for results in all_results.values() for r in results]
    if not flat_results:
        return ''

    plt.rcParams.update({
        'font.family': 'sans-serif',
        'axes.spines.top': False,
        'axes.spines.right': False,
    })
    COLOUR = '#667eea'
    svg_parts = []
    saved_paths = []

    prefix = file_prefix or 'search'

    def fig_to_svg(fig, filename=None):
        if save_dir and filename:
            if timestamp:
                name, ext = filename.rsplit('.', 1)
                filename = f'{name}_{timestamp}.{ext}'
            out_path = Path(save_dir) / filename
            out_path.parent.mkdir(parents=True, exist_ok=True)
            fig.savefig(out_path, format='svg', bbox_inches='tight')
            saved_paths.append(str(out_path))
        buf = io.StringIO()
        fig.savefig(buf, format='svg', bbox_inches='tight')
        plt.close(fig)
        return buf.getvalue()

    year_counts = Counter()
    matched_docs_by_year = defaultdict(set)
    matched_doc_ids = set()

    for r in flat_results:
        if not getattr(r, 'year', None):
            continue
        year = int(r.year)
        year_counts[year] += 1
        doc_id = str(getattr(r, 'document_id', '') or '')
        if doc_id:
            matched_docs_by_year[year].add(doc_id)
            matched_doc_ids.add(doc_id)

    docs_in_year = {}
    current_doc_prevalence_pct = {}

    docs_df = METADATA.get('documents')
    if docs_df is not None and len(docs_df) > 0:
        md = docs_df.copy()

        if 'File prefix' in md.columns and 'Year' in md.columns:
            md = md.dropna(subset=['File prefix', 'Year'])
            md['File prefix'] = pd.to_numeric(md['File prefix'], errors='coerce').astype('Int64')
            md['Year'] = pd.to_numeric(md['Year'], errors='coerce').astype('Int64')
            md = md.dropna(subset=['File prefix', 'Year']).copy()
            md['File prefix'] = md['File prefix'].astype(int).astype(str)
            md['Year'] = md['Year'].astype(int)

            docs_in_year = md.groupby('Year')['File prefix'].nunique().to_dict()

            if 'ISO Alpha-3 Code' in md.columns:
                country_docs = defaultdict(list)
                for row in md[['ISO Alpha-3 Code', 'Year', 'File prefix']].itertuples(index=False):
                    iso = str(row[0]).strip()
                    if not iso:
                        continue
                    country_docs[iso].append((int(row[1]), str(row[2]), int(row[2])))

                for iso in country_docs:
                    country_docs[iso].sort(key=lambda x: (x[0], x[2]))

                if year_counts:
                    years = sorted(year_counts.keys())
                    for y in years:
                        active_countries = 0
                        current_docs_with_hit = 0

                        for _iso, entries in country_docs.items():
                            current_doc = None
                            for doc_year, doc_id, _fp_num in entries:
                                if doc_year <= y:
                                    current_doc = doc_id
                                else:
                                    break

                            if current_doc is None:
                                continue

                            active_countries += 1
                            if current_doc in matched_doc_ids:
                                current_docs_with_hit += 1

                        if active_countries > 0:
                            current_doc_prevalence_pct[y] = (current_docs_with_hit / active_countries) * 100.0

    if year_counts:
        years = sorted(year_counts)
        counts = [year_counts[y] for y in years]

        docs_published = [docs_in_year.get(y, 0) for y in years]
        yearly_doc_prevalence_pct = [
            (len(matched_docs_by_year[y]) / docs_in_year[y] * 100.0) if docs_in_year.get(y, 0) > 0 else 0.0
            for y in years
        ]
        current_prevalence_series = [current_doc_prevalence_pct.get(y, float('nan')) for y in years]

        fig, (ax_raw, ax_norm) = plt.subplots(
            2, 1, figsize=(10, 7), sharex=True,
            gridspec_kw={'height_ratios': [2, 1.4]}
        )

        ax_raw.bar(years, counts, color=COLOUR, width=0.7)
        ax_raw.set_ylabel('Results')
        ax_raw.set_title('Results by year (raw counts)')
        ax_raw.grid(axis='y', alpha=0.2)

        line1, = ax_norm.plot(
            years, yearly_doc_prevalence_pct,
            color='#2e7d32', marker='o', linewidth=2,
            label='Yearly prevalence (% of docs published in year with >=1 hit)'
        )

        handles = [line1]
        labels = [line1.get_label()]

        if any(y in current_doc_prevalence_pct for y in years):
            line2, = ax_norm.plot(
                years, current_prevalence_series,
                color='#ef6c00', marker='s', linewidth=2,
                label='Current-doc prevalence (% of active countries)'
            )
            handles.append(line2)
            labels.append(line2.get_label())

        ax_norm.set_xlabel('Publication year')
        ax_norm.set_ylabel('Prevalence (%)')
        ax_norm.set_title('Normalized by corpus size and membership growth')
        ax_norm.set_xticks(years)
        ax_norm.tick_params(axis='x', rotation=45)
        ax_norm.grid(axis='y', alpha=0.3)
        ax_norm.legend(handles, labels, loc='upper left', fontsize=8)

        fig.tight_layout()
        svg_parts.append(fig_to_svg(fig, f'{prefix}_by_year.svg'))

        median_docs_year = int(np.median([v for v in docs_published if v > 0])) if any(v > 0 for v in docs_published) else 0
        svg_parts.append(
            '<p style="font-size:0.92em;color:#666;margin:4px 0 16px 0;">'
            '<strong>How to interpret this chart:</strong> '
            'The top panel is the raw number of matched segments per year, which can be pulled up '
            'by longer documents or repeated mentions within the same document. '
            'The lower panel therefore uses document-level prevalence ("at least one hit per document"). '
            'The green line asks: "Of documents published in this year, what share mention the topic at least once?" '
            'The orange line asks: "Among countries with at least one document by this year, what share of their '
            'current (latest-available) document mentions the topic at least once?" '
            'Use these two prevalence lines for trend comparison; use the raw bars for workload/volume context. '
            f'Median docs/year in corpus: {median_docs_year}.'
            '</p>'
        )

    country_counts = Counter(r.country for r in flat_results if r.country)
    if country_counts:
        top = country_counts.most_common(15)
        labels = [c for c, _ in reversed(top)]
        values = [v for _, v in reversed(top)]
        fig, ax = plt.subplots(figsize=(8, max(3, len(labels) * 0.35)))
        ax.barh(labels, values, color=COLOUR)
        ax.set_xlabel('Segments')
        ax.set_title('Top countries')

        x_max = max(values) if values else 1
        country_doc_counts = defaultdict(set)
        for r in flat_results:
            if r.country and getattr(r, 'document_id', None):
                country_doc_counts[r.country].add(str(r.document_id))
        for bar, lbl, val in zip(ax.patches, labels, values):
            n_docs = len(country_doc_counts.get(lbl, set()))
            ax.text(
                val + x_max * 0.015,
                bar.get_y() + bar.get_height() / 2,
                f'{val} / {n_docs}',
                va='center', ha='left', fontsize=7.5, color='#374151'
            )
        ax.set_xlim(right=x_max * 1.18)

        fig.tight_layout(rect=[0, 0.12, 1, 1])
        fig.text(
            0.5, 0.03,
            'Note: counts reflect document volume and length \u2014 countries publishing more or longer\n'
            'documents will appear higher (e.g. Japan publishes very lengthy annual documents).\n'
            'Bar labels show segments / documents.',
            ha='center', fontsize=9.5, color='#6b7280', style='italic'
        )
        svg_parts.append(fig_to_svg(fig, f'{prefix}_top_countries.svg'))

    cluster_sizes = []
    cluster_table = []                           # (label, results, most typical sentence) for the table under the chart
    _multi_topic = len(all_cluster_info) > 1
    for topic, ci in all_cluster_info.items():
        cd = ci.get('cluster_dict', {})
        names = ci.get('cluster_names', {})
        for cid, results in cd.items():
            label = names.get(cid, str(cid))
            if _multi_topic:                    # cluster numbers restart in every topic, so name the topic too
                label = f'{label} [{topic[:28]}{"…" if len(topic) > 28 else ""}]'
            cluster_sizes.append((label, len(results)))
            cluster_table.append((label, len(results), (ci.get('cluster_examples') or {}).get(cid, '')))
    if cluster_sizes:
        cluster_sizes.sort(key=lambda x: x[1])
        labels = [l for l, _ in cluster_sizes]
        values = [v for _, v in cluster_sizes]
        fig, ax = plt.subplots(figsize=(8, max(3, len(labels) * 0.32)))
        _pos = list(range(len(labels)))         # one position per bar: identical labels must not share a bar
        ax.barh(_pos, values, color=COLOUR)
        ax.set_yticks(_pos)
        ax.set_yticklabels(labels)
        ax.set_xlabel('Results')
        ax.set_title('Cluster overview')
        fig.tight_layout()
        svg_parts.append(fig_to_svg(fig, f'{prefix}_cluster_overview.svg'))
        from html import escape as _esc
        cluster_table.sort(key=lambda x: -x[1])
        _rows = ''.join(
            '<tr><td style="padding:3px 12px 3px 0;white-space:nowrap;">'
            + (f'<span class="nsddd-cl" tabindex="0" data-tip="{_esc(ex)}">{_esc(lb)}</span>' if ex else _esc(lb))
            + f'</td><td style="padding:3px 12px 3px 0;text-align:right;">{n}</td>'
            + f'<td style="padding:3px 0;color:#546e7a;">{_esc(ex[:110] + ("…" if len(ex) > 110 else ""))}</td></tr>'
            for lb, n, ex in cluster_table)
        svg_parts.append('<table style="font-family:sans-serif;font-size:13px;border-collapse:collapse;">'
                         '<tr style="text-align:left;color:#37474f;"><th style="padding:3px 12px 3px 0;">Cluster (hover for its most typical sentence)</th>'
                         '<th style="padding:3px 12px 3px 0;text-align:right;">Results</th><th style="padding:3px 0;">Most typical sentence</th></tr>'
                         + _rows + '</table>')

    if not svg_parts:
        return ''

    if saved_paths:
        print(f'  ✓ Saved {len(saved_paths)} SVG charts to {save_dir}/')

    html = '<div style="margin-top: 30px;">'
    html += '<h3 style="color: #546e7a; font-family: sans-serif;">📊 Visualisations</h3>'
    for svg in svg_parts:
        html += f'<div style="margin-bottom: 20px;">{svg}</div>'
    html += '</div>'
    return html

print('✅ Visualisation function loaded')



In [ ]:
%%capture

def create_search_interface():
    """Create the main search interface with all filtering options."""

    # Title
    _app_version = _read_local_version()
    _guide_pdf = 'NSDDD_Topic_Search_guide_v4.0.pdf'
    _guide_link = ''
    if Path(_guide_pdf).exists():
        _guide_url = os.environ.get('NSDDD_VOILA_BASE_URL', 'http://127.0.0.1:8867').rstrip('/') + '/voila/files/' + _guide_pdf
        _guide_link = (f'<p style="margin: 8px 0 0 0; font-size: 13px;"><a href="{_guide_url}" target="_blank" '
                       'style="color: #fff; text-decoration: underline;">Open the user guide (PDF)</a></p>')
    try:    # link to the feedback form; the app, data and system versions are filled in, nothing is sent until the user submits
        from _library.feedback import feedback_url, data_version
        _feedback_link = ('<p style="margin: 6px 0 0 0; font-size: 13px;"><a href="' + feedback_url(_app_version, data_version(MODEL_PATH)) + '" target="_blank" '
                          'style="color: #fff; text-decoration: underline;">Report a problem or suggest a feature</a> '
                          '<span style="opacity: 0.75; font-size: 12px;">(opens a short form in your browser; only the version numbers are filled in, and nothing is sent until you press Submit)</span></p>')
    except Exception:
        _feedback_link = ''
    try:    # what is actually loaded right now (documents and countries from the metadata, segments from the index)
        _n_docs, _n_countries = len(doc_df), doc_df['Country'].nunique()
        _n_segments = json.loads((Path(MODEL_PATH) / 'segment_encodings_metadata.json').read_text())['memmap_shape'][0]
        _loaded = f"{_n_docs:,} documents &middot; {_n_countries} countries &middot; {_n_segments:,} segments loaded"
    except Exception:
        _loaded = ''
    title = widgets.HTML(
        value=f"""
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    padding: 25px; border-radius: 10px; color: white; text-align: center;">
            <h1 style="margin: 0;">🌍 NSDDD Search Interface</h1>
            <p style="margin: 10px 0 0 0; opacity: 0.9;">Search with comprehensive filtering options</p>
            <p style="margin: 8px 0 0 0; font-size: 13px; font-weight: 600;">{_loaded}</p>
            {_guide_link}
            {_feedback_link}
            <p style="margin: 6px 0 0 0; opacity: 0.6; font-size: 12px;">v{_app_version}</p>
        </div>
        """
    )

    # Search configuration section
    search_header = widgets.HTML(
        value="<h3 style='margin: 20px 0 10px 0; color: #546e7a;'>🔍 Search Configuration</h3>"
    )

    topics_widget = widgets.Textarea(
        value='',
        placeholder='Enter search topics (one per line)\nExample:\ncyber threats\nclimate security',
        description='Topics:',
        layout=Layout(width='80%', height='100px')
    )

    search_threshold = widgets.FloatSlider(
        value=0.72,      # Default raised from 0.69 on 2026-10-04: below 0.72 most results are only loosely on topic (see the help text)
        min=0.50,
        max=0.90,
        step=0.005,      # Qwen3 scores are compact: 0.01 roughly halves or doubles the results
        readout_format='.3f',
        description='Search threshold:',
        style={'description_width': 'initial'},
        layout=Layout(width='60%')
    )

    try:                                        # is the small label model installed? (checked without loading it)
        from _library import cluster_labeller as _CLB
        _ai_ok, _ai_msg = _CLB.status(MODEL_PATH)
    except Exception:
        _ai_ok, _ai_msg = False, 'AI headings are not available in this installation'

    ai_during_widget = widgets.Checkbox(
        value=False, indent=False, disabled=not _ai_ok,
        description=('Write AI headings during the search (slower; off by default)' if _ai_ok
                     else 'AI headings are not available in this installation'),
        layout=Layout(width='auto', margin='0 0 6px 20px')
    )

    cluster_threshold = widgets.FloatSlider(
        value=0.78,
        min=0.50,
        max=0.95,
        step=0.01,
        readout_format='.2f',
        description='Cluster detail (higher = more, smaller clusters):',
        style={'description_width': 'initial'},
        layout=Layout(width='60%')
    )

    semantic_help_widget = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px;
                    background-color: #fff9e6; border-left: 3px solid #ffa726;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Semantic Search Tips:</b><br>
            • <b>How it works:</b> Uses AI to understand meaning, not just keywords<br>
            • <b>Best with:</b> Complete sentences or phrases (e.g., 'climate change poses security threats')<br>
            • <b>Why sentences:</b> Provides context that improves semantic matching accuracy<br>
            • <b>Search threshold:</b> Qwen3 scores are compact, so small changes matter. The default is 0.72. In a test of three topics, most results sit between 0.69 and 0.72, but for a broad topic such as &lsquo;climate change as a security threat&rsquo; only about one in ten of those was clearly on topic, against about three in four from 0.72 to 0.75 and nearly all above 0.75. Lower it to 0.69 for an exhaustive search (about 3,700 results for that topic against about 570 at 0.72); raise it to 0.75 for a tightly focused one<br>
            • <b>Example:</b> 'cyber attacks' finds 'digital threats', 'online security breaches', etc.<br>
            • <b>Use when:</b> Exploring concepts, finding related ideas, or researching topics
        </div>
    """)

    clustering_help_widget = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px;
                    background-color: #e8f5e9; border-left: 3px solid #4caf50;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Clusters (both search modes):</b><br>
            • <b>What they are:</b> results that say similar things are grouped, and the groups are numbered from the largest (Cluster 1) down. A cluster is a group of similar sentences, not a named topic<br>
            • <b>Telling them apart:</b> hover over a cluster label on a result to see that cluster&rsquo;s most typical sentence. The same sentence is in the &lsquo;Cluster example&rsquo; column of the export<br>
            • <b>Cluster detail:</b> sets how many groups are made. Raising it by 0.04 doubles the number of clusters and lowering it by 0.04 halves it, up to 60. At 0.78 about 4,000 results give about 20 clusters and about 60 results give 2 or 3<br>
            • <b>Important:</b> this setting only controls how results are grouped. It does not affect which results are returned<br>
            • <b>Headings:</b> clusters are numbered. Tick &lsquo;Write AI headings during the search&rsquo; to have a short heading written for every cluster as part of the search, or press &lsquo;Add AI headings&rsquo; under the Cluster overview once a search is done
        </div>
    """)

    # Search mode section
    mode_header = widgets.HTML(
        value="<h3 style='margin: 20px 0 10px 0; color: #546e7a;'>🔎 Search Mode</h3>"
    )

    keyword_search_widget = widgets.Checkbox(
        value=False,
        description='Use keyword search (regex) instead of semantic search',
        indent=False,
        style={'description_width': 'initial'},
        layout=Layout(width='max-content')
    )

    keyword_help_widget = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px;
                    background-color: #f0f4f8; border-left: 3px solid #4c6ef5;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Keyword Search Tips:</b><br>
            • <b>Wildcards:</b> Use <code>*</code> for any characters, <code>?</code> for single character<br>
            • <b>Phrases/N-grams:</b> Multi-word phrases work (e.g., <code>climate change</code>)<br>
            • <b>Case-insensitive:</b> Search ignores case by default<br>
            • <b>Word boundaries:</b> Matches whole words only<br>
            • <b>Special characters:</b> Most are automatically escaped<br>
            • <b>Original languages:</b> Keyword and Boolean searches also search the original-language text of the documents that have it, in whatever language you type. Type the word as the document spells it, with its accents and special letters (<code>zaštita</code> will not find <code>zastita</code>). Capitals do not matter. Use <code>OR</code> to search several languages at once, e.g. <code>cyber* OR ciber* OR кибер*</code><br>
            • <b>One language at a time:</b> A word matches only text in its own language, so <code>security</code> will not find &ldquo;Sicherheit&rdquo;. To find the same idea in every language, use semantic search. Each result shows its language, and the aligned text in the other language where there is one
        </div>
    """)

    boolean_mode_widget = widgets.Checkbox(
        value=False,
        description='Enable Boolean operators (AND, OR, NOT)',
        indent=False,
        style={'description_width': 'initial'},
        layout=Layout(width='max-content')
    )

    boolean_scoring_widget = widgets.Dropdown(
        options=[('Match count (default)', 'match_count'),
                 ('Average score', 'avg_score'),
                 ('Maximum score', 'max_score')],
        value='match_count',
        description='Boolean scoring:',
        style={'description_width': 'initial'},
        layout=Layout(width='400px')
    )

    lang_view_widget = widgets.Dropdown(
        options=[('All languages (English + original)', 'all'),
                 ('English only', 'en'),
                 ('Original languages only', 'orig')],
        value='all',
        description='Search languages:',
        style={'description_width': 'initial'},
        layout=Layout(width='400px')
    )

    count_once_widget = widgets.Checkbox(
        value=True,
        description='Count each passage once (merge an original-language match with the English match of the same passage)',
        style={'description_width': 'initial'},
        layout=Layout(width='max-content')
    )

    max_display_widget = widgets.Dropdown(
        options=[('5 documents', 5), ('10 documents', 10), ('25 documents', 25),
                 ('50 documents', 50), ('100 documents', 100)],
        value=5,
        description='Results to show:',
        style={'description_width': 'initial'},
        layout=Layout(width='400px')
    )

    language_help_widget = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px;
                    background-color: #eef7f1; border-left: 3px solid #0b8457;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Languages &amp; alignment.</b> The corpus holds documents in English and, where the
            source is not English, the <b>original language</b> alongside an English translation.
            Search is <b>cross-lingual</b> &mdash; a query in any language finds relevant passages in
            any language.<br>
            &bull; Each result shows the matching passage in English with its <b>original-language text</b>
            beneath (badge e.g. <i>NO &middot; Norsk (original)</i>); English-only documents show English.<br>
            &bull; When both the original and the English of one passage match, they are <b>counted as one
            result</b> (the English match is kept). Untick &lsquo;Count each passage once&rsquo; to count them
            separately; the total then reads &ldquo;X English + Y original-language&rdquo;. An original-language
            passage whose English did not match is kept and counted.<br>
            &bull; Original&harr;English <b>alignment is automatic and sentence-level</b>; it can be
            imperfect, so <b>low-confidence pairings are hidden</b> rather than shown incorrectly.<br>
            &bull; Use <b>&ldquo;&#8651; View side-by-side (parallel)&rdquo;</b> on a result (or the
            document viewer) to read the full original and English columns together.
        </div>
        """)

    boolean_help_widget = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px;
                    background-color: #f0f4f8; border-left: 3px solid #7c3aed;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Boolean Search Operators:</b><br>
            • <b>AND</b> (<code>&</code>), <b>OR</b> (<code>|</code>), <b>NOT</b> (<code>!</code>)<br>
            • <b>Parentheses</b>: Control precedence — <code>(cyber OR digital) AND threat</code><br>
            • <b>Quotes</b>: Exact phrases — <code>"climate change" AND security</code><br>
            • <b>Original languages</b>: Terms work in any language, with accents as written. Combine languages with OR: <code>(security OR Sicherheit OR sécurité) AND defen*</code>
        </div>
    """)

    fuzzy_matching_widget = widgets.Checkbox(
        value=False,
        description='Enable fuzzy matching' if FUZZY_AVAILABLE else 'Fuzzy matching unavailable (optional dependency missing)',
        disabled=not FUZZY_AVAILABLE,
        indent=False,
        style={'description_width': 'initial'},
        layout=Layout(width='max-content')
    )

    smart_boundaries_widget = widgets.Checkbox(
        value=True,
        description='Smart phrase matching (handles hyphens/punctuation)',
        indent=False,
        style={'description_width': 'initial'},
        layout=Layout(width='max-content')
    )

    fuzzy_threshold_widget = widgets.IntSlider(
        value=80,
        min=50,
        max=100,
        step=5,
        description='Fuzzy threshold:',
        style={'description_width': 'initial'},
        layout=Layout(width='60%')
    )

    fuzzy_help_widget = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px;
                    background-color: #f0f4f8; border-left: 3px solid #4c6ef5;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Fuzzy Matching Tips:</b> Matches approximate terms (catches typos). 80% is recommended.
        </div>
    """)

    smart_boundaries_help_widget = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px;
                    background-color: #f0f4f8; border-left: 3px solid #4c6ef5;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Smart Phrase Matching:</b> Handles variation like "cyber-security" vs "cyber security".
        </div>
    """)

    context_window_widget = widgets.IntSlider(
        value=1,
        min=0,
        max=5,
        step=1,
        description='Context window:',
        style={'description_width': 'initial'},
        layout=Layout(width='60%')
    )

    geo_header = widgets.HTML(value="<h3 style='margin: 20px 0 10px 0; color: #546e7a;'>🌐 Geographic Filters</h3>")

    regions_widget = widgets.SelectMultiple(
        options=METADATA.get('unique_regions', []),
        description='Regions:',
        layout=Layout(width='60%', height='100px')
    )
    subregions_widget = widgets.SelectMultiple(
        options=METADATA.get('unique_subregions', []),
        description='Subregions:',
        layout=Layout(width='60%', height='120px')
    )

    # Initialize country lookup
    country_lookup = CountryLookup()
    all_countries = country_lookup.list_all_countries(show_region=False)
    country_options = [(f"{num:3d}. {iso:3s} {name}", iso) for num, iso, name, _ in all_countries]

    countries_widget = widgets.SelectMultiple(
        options=country_options,
        description='Countries:',
        layout=Layout(width='60%', height='200px')
    )

    filter_selection_help = widgets.HTML(value="""
        <div style="margin: 5px 0 15px 0; padding: 8px 12px;
                    background-color: #e8f5e9; border-left: 3px solid #66bb6a;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Selecting and deselecting filter options (applies to all lists below):</b><br>
            • Click to select; click again to deselect<br>
            • Mac: Cmd+Click; Windows: Ctrl+Click — toggles individual items<br>
            • Shift+Click selects a range<br>
            • Leave any list empty to include all options
        </div>
    """)

    org_header = widgets.HTML(value="<h3 style='margin: 20px 0 10px 0; color: #546e7a;'>🤝 International Organizations</h3>")
    organizations_widget = widgets.SelectMultiple(
        options=sorted(METADATA.get('organizations', {}).keys()),
        description='Organizations:',
        layout=Layout(width='60%', height='150px')
    )
    membership_when_widget = widgets.Dropdown(
        options=[('Count members as of the year each document was published', 'published'),
                 ('Count members as of today', 'now')],
        value='published',
        description='Membership:',
        style={'description_width': 'initial'},
        layout=Layout(width='70%')
    )
    membership_help = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px; background-color: #e3f2fd; border-left: 3px solid #1e88e5;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>As of the year published</b> (the default) asks whether the country was a member in the year of the document. A Finnish document of 2015 is then not NATO,
            and a British document of 2015 is EU.<br>
            <b>As of today</b> keeps every document from a country that is a member now, whenever it was written: a Finnish document of 2015 counts as NATO,
            and a British document of 2015 is left out because the United Kingdom has left the EU. A country counts from its year of joining and stops in its year of leaving, so a document written in the year
            of accession may still pre-date it.<br>
            <b>Suspended</b> members still count as members. Observers, dialogue partners and associate members do not.
            The State Union of Serbia and Montenegro (2003 to 2006) takes Serbia&rsquo;s memberships up to 2006.
        </div>
    """)

    econ_header = widgets.HTML(
        value="<h3 style='margin: 20px 0 10px 0; color: #546e7a; font-weight: bold;'>💰 Economic & Political Filters</h3>"
    )

    time_based_note = widgets.HTML(
        value="""
        <div style='margin: 10px 0; padding: 12px;
                    background-color: #e3f2fd;
                    border-left: 5px solid #1565c0;
                    color: #0d47a1;
                    font-size: 14px;'>
            <strong>Note:</strong> Income group, democracy status and aid status use the latest available classification. Organisation membership can be counted as of today or as of each document&rsquo;s year (see Organizations).
        </div>
        """
    )
    income_widget = widgets.SelectMultiple(options=['High', 'Upper-middle', 'Lower-middle', 'Low'], description='Income groups:', layout=Layout(width='60%', height='100px'))
    freedom_widget = widgets.SelectMultiple(options=['Free', 'Partly Free', 'Not Free'], description='Democracy:', layout=Layout(width='60%', height='80px'))
    oda_widget = widgets.SelectMultiple(options=['ODA_Recipient', 'Non_Recipient'], description='ODA status:', layout=Layout(width='60%', height='80px'))
    small_states_widget = widgets.SelectMultiple(options=['Small States', 'SSF', 'UN SIDS'], description='Small States:', layout=Layout(width='60%', height='80px'))
    huntington_widget = widgets.SelectMultiple(options=METADATA.get('huntington_civilizations', []), description='Huntington', layout=Layout(width='60%', height='120px'))

    doc_header = widgets.HTML(value="<h3 style='margin: 20px 0 10px 0; color: #546e7a;'>📄 Document Filters</h3>")
    doc_types_widget = widgets.SelectMultiple(options=['NSS', 'WP', 'DD', 'TA'], description='Doc types:', layout=Layout(width='60%', height='100px'))
    year_range_widget = widgets.IntRangeSlider(value=[1987, 2026], min=1987, max=2026, step=1, description='Year range:', layout=Layout(width='60%'))
    most_recent_widget = widgets.Checkbox(value=False, description='Most recent document per country only', layout=Layout(width='max-content'))

    # CSV Export widgets
    export_csv_widget = widgets.Checkbox(
        value=not LOW_POWER,
        description='Export results to a file  (see outputs/ folder)',
        style={'description_width': 'initial'},
        layout=Layout(width='max-content')
    )

    export_format_widget = widgets.Dropdown(
        options=[('Excel workbook (.xlsx): clickable links, opens correctly in every language version of Excel', 'xlsx'),
                 ('CSV with plain web addresses: links are not clickable. If Excel puts everything in one column, use Data > From Text/CSV', 'csv')],
        value='xlsx',
        description='File type:',
        style={'description_width': 'initial'},
        layout=Layout(width='80%')
    )

    export_format_help = widgets.HTML(value="""
        <div style="margin: 5px 0 10px 20px; padding: 8px 12px; background-color: #e3f2fd; border-left: 3px solid #1e88e5;
                    color: #1a202c; font-size: 12px; line-height: 1.5;">
            <b>Excel workbook (recommended).</b> One file with three sheets: Results, Search settings and Normalisation context.
            The Link column holds real hyperlinks, which work whatever your regional settings.<br>
            <b>CSV.</b> Two plain-text files. The Link column holds each web address as text, to paste into a browser.
            Excel set to Danish, German, French or another continental European language expects semicolons, so a CSV opened by
            double-clicking lands in a single column there. Open it through Data, then From Text/CSV, and choose the comma as the delimiter.<br>
            <b>Either way,</b> the links open only while NSDDD is running on the computer that made the file.
        </div>
    """)

    iterative_export_widget = widgets.Checkbox(
        value=True,
        description='Use iterative filenames (add timestamp to avoid overwriting)',
        style={'description_width': 'initial'},
        layout=Layout(width='max-content')
    )

    export_warning = widgets.HTML(
        value='<div style="color: #d32f2f; font-size: 0.9em; margin-top: 5px;">⚠️ Searches with the same name will overwrite previous files unless iterative filenames are enabled.</div>',
        layout=Layout(width='100%', margin='5px 0')
    )

    search_button = Button(description='🔍 Search', button_style='success', layout=Layout(width='200px', height='50px'), style={'button_color': '#4CAF50', 'font_weight': 'bold'})
    clear_filters_button = Button(description='✕ Clear All Filters', button_style='warning', layout=Layout(width='200px', height='50px'))
    output = widgets.Output()
    scroll_widget = widgets.HTML(value='<span id="ns-scroll-trigger" data-n="0"></span>')
    _scroll_n = [0]

    def trigger_scroll():
        _scroll_n[0] += 1
        scroll_widget.value = '<span id="ns-scroll-trigger" data-n="' + str(_scroll_n[0]) + '"></span>'

    output_anchor = widgets.HTML('<div id="search-output-anchor" style="height:0;padding:0;margin:0;"></div>')
    charts_enabled_widget = widgets.Checkbox(value=not LOW_POWER, description='Generate charts (untick to save time and memory)')
    refresh_button = widgets.Button(description='Update charts and exports', icon='refresh')
    charts_widget = widgets.HTML(value='')
    norm_output = widgets.Output()
    thinking_indicator = widgets.HTML(value='''<div style="display:none;"></div>''')

    _last = {}
    ai_info = widgets.HTML(value="""
        <div style="margin: 14px 0 8px 0; padding: 14px 18px; background: #fff8e1; border-left: 5px solid #f9a825;
                    color: #1a202c; font-size: 14px; line-height: 1.6; border-radius: 4px;">
            <div style="font-size: 16px; font-weight: 700; margin-bottom: 6px;">Add AI headings to the clusters</div>
            <p style="margin: 0 0 8px 0;"><b>What you have now.</b> The clusters are numbered, largest first (Cluster 1, Cluster 2 and so on).
            A number says nothing about what a cluster contains, so hover over a cluster label (on any result, or in the table under the Cluster overview chart) to read the cluster&rsquo;s most typical sentence.
            The same sentence is in the &lsquo;Cluster example&rsquo; column of the exported file.</p>
            <p style="margin: 0 0 8px 0;"><b>What the button does.</b> It asks a small language model that runs on this computer (Qwen3-0.6B, about 520 MB)
            to read six representative sentences from each cluster and to write a heading of two to eight words, such as
            &lsquo;Risks in cyberspace&rsquo;. Nothing is sent over the internet.</p>
            <p style="margin: 0 0 8px 0;"><b>Why it is useful.</b> Headings let you scan a large set of results quickly, compare clusters
            across different searches and decide which clusters to read first. They also make a figure or an exported table easier to follow.</p>
            <p style="margin: 0 0 8px 0;"><b>What to bear in mind.</b> A heading is a machine-written summary of a handful of sentences, not a finding.
            It can be vague or, occasionally, misleading, so check it against the example sentence and the segments before you rely on it.
            If the model cannot write an acceptable heading for a cluster, that cluster keeps its number. The tool does not substitute keyword lists.</p>
            <p style="margin: 0;"><b>How long it takes.</b> Roughly two seconds per cluster on a mid-range laptop, so a search with 60 clusters can take a couple of minutes.
            The results above and the exported file are then updated with the headings. To have headings written automatically, tick &lsquo;Write AI headings during the search&rsquo; beside the cluster detail slider before you search.</p>
        </div>
    """)
    ai_button = Button(description='Add AI headings', button_style='info', icon='tags', layout=Layout(width='220px', height='42px'))
    ai_status = widgets.HTML(value='')
    ai_panel = VBox([HBox([ai_button, ai_status], layout=Layout(align_items='center', gap='14px')), ai_info],
                    layout=Layout(display='none', margin='10px 0'))

    def _left_numbered(infos):
        ids = sorted({int(c.split('_')[-1]) for ci in infos for c in (ci.get('label_inputs') or {})
                      if c not in ci.get('ai_named', []) and c.split('_')[-1].isdigit()})
        return ', '.join('Cluster ' + str(i) for i in ids[:12]) + (' and ' + str(len(ids) - 12) + ' more' if len(ids) > 12 else '')

    def _refresh_ai_panel():
        infos = list(_last.get('info', {}).values())
        total = sum(len(ci.get('label_inputs') or {}) for ci in infos)
        ai_panel.layout.display = '' if total else 'none'
        if not total:
            return
        named = sum(len(ci.get('ai_named', [])) for ci in infos)
        if not _ai_ok:
            ai_button.disabled = True
            ai_status.value = '<span style="color:#b71c1c;">' + escape(_ai_msg) + '. The clusters stay numbered.</span>'
        elif any(ci.get('label_source', 'numbers') != 'numbers' for ci in infos) or any('ai_named' in ci for ci in infos):
            ai_button.disabled = True
            ai_status.value = ('<span style="color:#1b5e20;">AI headings added to ' + str(named) + ' of ' + str(total) + ' clusters.'
                               + ('' if named == total else ' Left numbered: ' + _left_numbered(infos) + ', where the model could only restate the search topic or no acceptable heading was found. Hover over the label to read the cluster&rsquo;s example sentence.')
                               + ' Search again to start from numbers.</span>')
        else:
            ai_button.disabled = False
            ai_status.value = ''

    def on_ai_click(b):
        if not _last:
            return
        b.disabled = True
        search_button.disabled = True
        bottom_search_button.disabled = True
        try:
            ai_status.value = '<span style="color:#0d47a1;">Loading the label model (the first time takes a little while)...</span>'

            def _progress(named, total, done, n):
                ai_status.value = ('<span style="color:#0d47a1;">Writing headings: cluster ' + str(min(done + 1, n)) + ' of ' + str(n)
                                   + ', ' + str(named) + ' named so far...</span>')
            apply_ai_cluster_labels(_last['results'], _last['info'], MODEL_DICT, progress=_progress)
            with output:
                output.clear_output()
                _show(_last['results'], _last['info'], _last['config'])
        except Exception as e:
            ai_status.value = '<span style="color:#b71c1c;">Could not add headings: ' + escape(str(e)) + '</span>'
            b.disabled = False
            return
        finally:
            search_button.disabled = False
            bottom_search_button.disabled = False
        _refresh_ai_panel()

    ai_button.on_click(on_ai_click)

    def _show(all_results, all_cluster_info, config):
        """Write the export, then draw the results, charts and normalisation panel. Call inside `with output:`."""
        _ts = _last.setdefault('ts', datetime.now().strftime('%Y%m%d_%H%M%S'))
        try:
            exported_files = export_search_results(all_results, all_cluster_info, config,
                                enable_export=export_csv_widget.value,
                                iterative_names=iterative_export_widget.value, timestamp=_ts)
        except OSError as _e:                   # on Windows a workbook open in Excel is locked and cannot be rewritten
            exported_files = None
            display(HTML('<div style="margin:8px 0;padding:10px 14px;background:#fff3e0;border-left:4px solid #ef6c00;color:#1a202c;font-size:13px;">'
                         '<b>The results below are up to date, but the export file was not rewritten.</b> '
                         'It is probably open in Excel or another program, which locks it on Windows. '
                         'Close it and search again to write a new copy. (' + escape(str(_e)) + ')</div>'))
        if export_csv_widget.value and exported_files:
            for _fp in exported_files:
                if '_norm_context' in Path(_fp).name:
                    continue
                _btn = widgets.Button(
                    description=Path(_fp).name,
                    button_style='',
                    icon='download',
                    layout=widgets.Layout(width='auto', margin='8px 0 4px 0')
                )
                _btn.style.button_color = '#1565c0'
                _btn.style.text_color = 'white'
                def _on_open(b, fp=_fp):
                    open_file(fp)
                _btn.on_click(_on_open)
                display(_btn)
        display(HTML('<style>.nsddd-cl{position:relative;cursor:help;border-bottom:1px dotted #607d8b;}'
                     '.nsddd-cl:hover::after,.nsddd-cl:focus::after{content:attr(data-tip);position:absolute;left:0;top:1.7em;z-index:10000;'
                     'width:max-content;max-width:min(420px,80vw);white-space:normal;background:#263238;color:#fff;padding:8px 10px;border-radius:6px;'
                     'font-size:12px;font-weight:400;line-height:1.45;box-shadow:0 3px 10px rgba(0,0,0,.35);pointer-events:none;}</style>'))
        for ci in all_cluster_info.values():
            if ci.get('clustering_note'):
                display(HTML('<p>' + escape(ci['clustering_note']) + '</p>'))
        display_batch_results(all_results, all_cluster_info, config, max_per_topic=max_display_widget.value)
        # Derive a safe file prefix from the first topic
        _first_topic = next(iter(all_results), 'search')
        _file_prefix = re.sub(r'_+', '_', re.sub(r'[^\w ]', '_', _first_topic)).strip().replace(' ', '_')[:50].strip('_') or 'search'
        _save_dir = 'outputs' if export_csv_widget.value else None
        _svg_timestamp = _ts if iterative_export_widget.value and export_csv_widget.value else None
        charts_widget.value = (generate_search_visualisations(all_results, all_cluster_info, save_dir=_save_dir, file_prefix=_file_prefix, timestamp=_svg_timestamp)
                               if charts_enabled_widget.value else '')
        norm_output.clear_output()
        with norm_output:
            if export_csv_widget.value and exported_files:
                _norm_csv_files = [_fp for _fp in exported_files
                                   if '_norm_context' in Path(_fp).name]
                for _fp in _norm_csv_files:
                    _nc_btn = widgets.Button(
                        description=Path(_fp).name,
                        button_style='',
                        icon='download',
                        layout=widgets.Layout(width='auto', margin='8px 0 4px 0')
                    )
                    _nc_btn.style.button_color = '#1565c0'
                    _nc_btn.style.text_color = 'white'
                    def _on_nc_open(b, fp=_fp):
                        open_file(fp)
                    _nc_btn.on_click(_on_nc_open)
                    display(_nc_btn)
            display(HTML(_build_norm_panel_html(all_results, eligible=_eligible_country_stats(config.filters))))

    def refresh_outputs(b):
        if _last.get('results'):
            with output:
                output.clear_output()
                _last['config'].export_format = export_format_widget.value
                _show(_last['results'], _last['info'], _last['config'])
    refresh_button.on_click(refresh_outputs)

    def on_search_click(b):
        global LANG_VIEW
        LANG_VIEW = lang_view_widget.value
        search_button.disabled = True
        bottom_search_button.disabled = True
        thinking_indicator.value = '''
        <div style="display:flex;align-items:center;gap:12px;margin:12px 0 14px 0;padding:12px 16px;background:#eef6ff;border-left:4px solid #1565c0;border-radius:8px;color:#0d47a1;">
            <div style="width:16px;height:16px;border:3px solid #bbdefb;border-top-color:#1565c0;border-radius:50%;animation: nsddd-spin 0.9s linear infinite;"></div>
            <div>
                <div style="font-weight:600;">Searching the corpus</div>
                <div style="font-size:12px;opacity:0.9;">Still working<span class="nsddd-thinking-dots"><span>.</span><span>.</span><span>.</span></span></div>
            </div>
        </div>
        <style>
        @keyframes nsddd-spin { from { transform: rotate(0deg); } to { transform: rotate(360deg); } }
        @keyframes nsddd-dot-fade { 0%, 80%, 100% { opacity: 0.25; } 40% { opacity: 1; } }
        .nsddd-thinking-dots span { display:inline-block; min-width:0.35em; animation: nsddd-dot-fade 1.2s infinite ease-in-out; }
        .nsddd-thinking-dots span:nth-child(2) { animation-delay: 0.15s; }
        .nsddd-thinking-dots span:nth-child(3) { animation-delay: 0.3s; }
        </style>
        '''
        try:
            # Scroll immediately on click so users see the output panel while search runs.
            trigger_scroll()

            charts_widget.value = ''
            norm_output.clear_output()
            _last.clear()
            ai_panel.layout.display = 'none'
            with output:
                output.clear_output()
                if MODEL_DICT is None:
                    error_html = '<div style="color: #c62828;">⚠️ Model not loaded.</div>'
                    if MODEL_LOAD_ERROR:
                        error_html += (
                            '<pre style="white-space: pre-wrap; background: #fff5f5; color: #7f1d1d; '
                            'padding: 12px; border: 1px solid #fecaca; border-radius: 6px; margin-top: 8px;">'
                            + escape(MODEL_LOAD_ERROR) + '</pre>'
                        )
                    display(HTML(error_html))
                    return

                topics = [t.strip() for t in topics_widget.value.split('\n') if t.strip()]
                if not topics:
                    display(HTML('<div style="color: #c62828;">⚠️ Please enter a topic.</div>'))
                    return

                if not keyword_search_widget.value and not boolean_mode_widget.value:
                    initialize_encoder()
                if not keyword_search_widget.value and not boolean_mode_widget.value and ENCODER is None:
                    error_html = '<div style="color: #c62828;">⚠️ Semantic search encoder is not loaded.</div>'
                    error_html += '<div style="margin-top: 6px;">You can still use keyword search, or fix the Python packages in this environment.</div>'
                    if ENCODER_LOAD_ERROR:
                        error_html += (
                            '<pre style="white-space: pre-wrap; background: #fff5f5; color: #7f1d1d; padding: 12px; border: 1px solid #fecaca; border-radius: 6px; margin-top: 8px;">'
                            + escape(ENCODER_LOAD_ERROR) + '</pre>'
                        )
                    display(HTML(error_html))
                    return

                filters = MetadataFilters(
                    regions=list(regions_widget.value),
                    subregions=list(subregions_widget.value),
                    countries=list(countries_widget.value),
                    organizations=list(organizations_widget.value),
                    membership_at_publication=(membership_when_widget.value == 'published'),
                    income_groups=list(income_widget.value),
                    freedom_house_status=list(freedom_widget.value),
                    oda_status=list(oda_widget.value),
                    small_states_groups=list(small_states_widget.value),
                    huntington_civilizations=list(huntington_widget.value),
                    doc_types=list(doc_types_widget.value),
                    year_min=year_range_widget.value[0],
                    year_max=year_range_widget.value[1],
                    most_recent_only=most_recent_widget.value
                )

                config = SearchConfig(
                    topics=topics,
                    search_threshold=round(search_threshold.value, 3),
                    cluster_threshold=round(cluster_threshold.value, 3),
                    export_format=export_format_widget.value,
                    filters=filters,
                    keyword_search=keyword_search_widget.value,
                    boolean_mode=boolean_mode_widget.value,
                    boolean_scoring=boolean_scoring_widget.value,
                    fuzzy_matching=fuzzy_matching_widget.value,
                    fuzzy_threshold=fuzzy_threshold_widget.value,
                    smart_boundaries=smart_boundaries_widget.value,
                    context_window=context_window_widget.value,
                    count_passages_once=count_once_widget.value
                )

                try:
                    all_results, all_cluster_info = run_metadata_batch_search(config, MODEL_DICT, ENCODER)
                    _last.clear()
                    _last.update(results=all_results, info=all_cluster_info, config=config)
                    if ai_during_widget.value and _ai_ok:
                        _spin = thinking_indicator.value
                        def _ai_progress(named, total, done, n):
                            thinking_indicator.value = (_spin.replace('Searching the corpus', 'Writing AI headings for the clusters')
                                .replace('Still working', 'Cluster ' + str(min(done + 1, n)) + ' of ' + str(n) + ', ' + str(named) + ' named so far'))
                        try:
                            apply_ai_cluster_labels(all_results, all_cluster_info, MODEL_DICT, progress=_ai_progress)
                        except Exception as _e:               # the search still succeeds; the clusters keep their numbers
                            display(HTML('<div style="margin:8px 0;padding:10px 14px;background:#fff3e0;border-left:4px solid #ef6c00;color:#1a202c;font-size:13px;">'
                                         '<b>AI headings could not be written, so the clusters keep their numbers.</b> (' + escape(str(_e)) + ')</div>'))
                    _show(all_results, all_cluster_info, config)
                    _refresh_ai_panel()
                except Exception as e:
                    import traceback
                    print(f"Error: {e}")
                    traceback.print_exc()
        finally:
            search_button.disabled = False
            bottom_search_button.disabled = False
            thinking_indicator.value = '<div style="display:none;"></div>'

        # Second trigger ensures viewport settles on output after rendering completes.
        trigger_scroll()

    search_button.on_click(on_search_click)

    def on_clear_filters_click(b):
        regions_widget.value = ()
        subregions_widget.value = ()
        countries_widget.value = ()
        organizations_widget.value = ()
        membership_when_widget.value = 'published'
        income_widget.value = ()
        freedom_widget.value = ()
        oda_widget.value = ()
        small_states_widget.value = ()
        huntington_widget.value = ()
        doc_types_widget.value = ()
        year_range_widget.value = [1987, 2026]
        most_recent_widget.value = False

    clear_filters_button.on_click(on_clear_filters_click)

    bottom_search_button = Button(description='🔍 Search', button_style='success', layout=Layout(width='200px', height='50px'), style={'button_color': '#4CAF50', 'font_weight': 'bold'})
    bottom_clear_button = Button(description='✕ Clear All Filters', button_style='warning', layout=Layout(width='200px', height='50px'))
    bottom_search_button.on_click(on_search_click)
    bottom_clear_button.on_click(on_clear_filters_click)

    toggle_js = widgets.HTML(value="""
        <script>
        (function() {
            function addToggle(sel) {
                if (sel._toggleDone) return;
                sel._toggleDone = true;
                sel.addEventListener('mousedown', function(e) {
                    if (e.target.tagName !== 'OPTION') return;
                    if (e.ctrlKey || e.metaKey || e.shiftKey) return;
                    e.preventDefault();
                    e.target.selected = !e.target.selected;
                    sel.dispatchEvent(new Event('change', {bubbles: true}));
                });
            }
            function init() {
                document.querySelectorAll('select[multiple]').forEach(addToggle);
                new MutationObserver(function() {
                    document.querySelectorAll('select[multiple]').forEach(addToggle);
                }).observe(document.documentElement, {childList: true, subtree: true});
            }
            if (document.readyState === 'loading') {
                document.addEventListener('DOMContentLoaded', init);
            } else {
                setTimeout(init, 500);
            }
        })();
        </script>
    """)

    interface = VBox([
        title, search_header, topics_widget, search_threshold, cluster_threshold, ai_during_widget,
        HBox([search_button, clear_filters_button], layout=Layout(gap='10px')),
        semantic_help_widget, clustering_help_widget, mode_header,
        keyword_search_widget, keyword_help_widget, fuzzy_matching_widget,
        fuzzy_threshold_widget, fuzzy_help_widget, smart_boundaries_widget,
        boolean_mode_widget, boolean_scoring_widget, boolean_help_widget,
        smart_boundaries_help_widget, context_window_widget,
        filter_selection_help,
        lang_view_widget,
        count_once_widget,
        max_display_widget,
        language_help_widget,
        geo_header,
        regions_widget, subregions_widget, countries_widget, org_header, organizations_widget, membership_when_widget, membership_help,
        econ_header, time_based_note, income_widget, freedom_widget,
        oda_widget, small_states_widget, huntington_widget, doc_header,
        doc_types_widget, year_range_widget, most_recent_widget,
        charts_enabled_widget,
        refresh_button,
        export_csv_widget,
        export_format_widget,
        export_format_help,
        iterative_export_widget,
        export_warning,
        HBox([bottom_search_button, bottom_clear_button], layout=Layout(gap='10px', margin='10px 0')),
        thinking_indicator,
        toggle_js,
        scroll_widget,
        output_anchor,
        output,
        charts_widget,
        ai_panel,
        norm_output
    ])

    return interface

print('✅ Interface functions defined')


In [ ]:
%%capture
# Example: Programmatic search
# This cell demonstrates how to run searches programmatically

# Define search configuration
example_filters = MetadataFilters(
    regions=['Asia'],
    organizations=['ASEAN'],
    year_min=2010,
    year_max=2026
)

example_config = SearchConfig(
    topics=['cyber security', 'maritime security'],
    search_threshold=0.72,
    filters=example_filters
)

print('Example configuration:')
print(f'Topics: {example_config.topics}')
print(f'Filters: {example_config.filters.get_summary()}')

# To run the search, you would call your search function here
# results = run_search(example_config, MODEL_DICT, ENCODER)

In [ ]:
%%capture
# Core similarity search with metadata filtering and clustering
def run_similarity_search(topic: str, config: SearchConfig, model_dict, encoder) -> Tuple[List[SearchResult], Dict]:
    """Run similarity search with metadata filtering and clustering."""
    if not encoder:
        print('❌ Encoder not loaded. Please initialize model first.')
        return [], {}

    # Get filtered documents
    doc_df = METADATA['documents']
    all_doc_ids = doc_df['File prefix'].astype(str).tolist()
    filtered_doc_ids = apply_metadata_filters(all_doc_ids, config.filters)

    if not filtered_doc_ids:
        print('⚠️  No documents match the filter criteria')
        return [], {}

    # Filter segments to selected documents
    encoded_segments = model_dict['encoded_segments']
    segment_encodings = model_dict['segment_encodings']
    segments_dict = model_dict['segments_dict']
    documents_dict = model_dict['documents_dict']
    countries_dict = model_dict['countries_dict']

    # Build filtered rows (a set: membership in a list was slow over 1.16 M segments)
    if model_dict.get('_doc_of_row') is None:      # document number of every index row, built once
        model_dict['_doc_of_row'] = np.array([int(s.split('/')[0]) for s in encoded_segments], dtype=np.int32)
    filtered_indices = np.nonzero(np.isin(model_dict['_doc_of_row'],
                                          np.array([int(d) for d in filtered_doc_ids], dtype=np.int32)))[0]
    if len(filtered_indices) == 0:
        print('⚠️  No segments found in filtered documents')
        return [], {}

    # Encode query (normalised; the stored vectors are normalised too)
    query_vector = np.asarray(encoder.encode(topic), dtype=np.float32).flatten()
    query_vector = query_vector / max(float(np.linalg.norm(query_vector)), 1e-10)

    # Score eligible rows in bounded blocks; avoid a full-index float32 copy.
    _cos = np.empty(len(filtered_indices), dtype=np.float32)
    for _s in range(0, len(filtered_indices), 25_000):
        _rows = filtered_indices[_s:_s + 25_000]
        _cos[_s:_s + len(_rows)] = np.asarray(segment_encodings[_rows], dtype=np.float32) @ query_vector
    if model_dict.get('short_mask') is not None:
        _cos -= model_dict.get('short_penalty', 0.05) * model_dict['short_mask'][filtered_indices]
    # Similarity on the 3.5 scale: 1 - (arccos(cosine) / π)
    similarities = 1.0 - (np.arccos(np.clip(_cos, -1.0, 1.0)) / np.pi)

    # Filter by threshold
    valid_mask = similarities >= config.search_threshold
    valid_segments = [encoded_segments[i] for i in filtered_indices[valid_mask]]
    valid_similarities = similarities[valid_mask]

    if len(valid_segments) == 0:
        return [], {}

    # Sort by similarity score (descending)
    segment_similarity_pairs = list(zip(valid_segments, valid_similarities))
    segment_similarity_pairs.sort(key=lambda x: x[1], reverse=True)

    # Apply max results limit if set
    if config.max_results_per_topic is not None:
        segment_similarity_pairs = segment_similarity_pairs[:config.max_results_per_topic]

    # Create search results
    results = []
    for segment_id, similarity in segment_similarity_pairs:
        doc_id = segment_id.split('/')[0]

        # Get segment text
        segment_data = segments_dict.get(segment_id, {})
        segment_text = get_segment_text_safe(segment_data)

        if not segment_text:
            continue

        # Get document metadata
        doc_data = documents_dict.get(doc_id, {})
        country_iso = doc_data.get('iso', '')

        # Get country name from countries_dict
        country_info = countries_dict.get(country_iso, {})
        country_name = country_info.get('country', doc_data.get('country', 'Unknown'))

        # Get context if requested
        context_before, context_after = get_context_window(
            segment_id, model_dict, config.context_window
        )

        result = SearchResult(
            segment_id=segment_id,
            text=segment_text,
            score=float(similarity),
            document_id=doc_id,
            document_name=doc_data.get('document', doc_data.get('name', '')),
            country=country_name,
            country_iso=country_iso,
            year=int(doc_data.get('year', 0)) if str(doc_data.get('year', '')).isdigit() else 0,
            doc_type=doc_data.get('type', ''),
            context_before=context_before,
            context_after=context_after
        )

        # Enrich with metadata
        result = enrich_result_with_metadata(result)
        results.append(result)

    results, _merged = _count_once(results, config)

    # Perform clustering if we have multiple results
    cluster_info_dict = {}
    if len(results) > 1:
        try:
            cluster_info_dict = cluster_search_results_internal(
                results, model_dict, config.cluster_threshold, search_query=topic
            )
        except Exception as e:
            print(f'⚠️  Clustering error: {str(e)}')
            # Fallback: all singletons
            cluster_info_dict = {
                'cluster_dict': {'singletons': list(range(len(results)))},
                'num_clusters': 0,
                'num_singletons': len(results),
                'cluster_names': {'singletons': 'Unclustered'}
            }
            for result in results:
                result.cluster_id = 'singletons'
                result.cluster_name = 'Unclustered'
    elif len(results) == 1:
        # Single result handling
        cluster_info_dict = {
            'cluster_dict': {'singletons': [0]},
            'num_clusters': 0,
            'num_singletons': 1,
            'cluster_names': {'singletons': 'Unclustered'}
        }
        results[0].cluster_id = 'singletons'
        results[0].cluster_name = 'Unclustered'

    cluster_info_dict['pairs_merged'] = _merged
    return results, cluster_info_dict

# Keyword search function
def run_keyword_search(keyword: str, config: SearchConfig, model_dict) -> Tuple[List[SearchResult], Dict]:
    """Run keyword search with metadata filtering and optional fuzzy matching."""
    if config.fuzzy_matching and not FUZZY_AVAILABLE:
        raise ValueError('Fuzzy matching is unavailable; install fuzzywuzzy or use exact keyword search.')
    print(f'🔍 Searching for keyword: "{keyword}"')

    # Get filtered documents
    doc_df = METADATA['documents']
    all_doc_ids = doc_df['File prefix'].astype(str).tolist()
    filtered_doc_ids = apply_metadata_filters(all_doc_ids, config.filters)

    if not filtered_doc_ids:
        print('⚠️  No documents match the filter criteria')
        return [], {}

    # Create keyword pattern
    pattern = create_keyword_pattern(keyword, smart_boundaries=config.smart_boundaries)
    results = []

    segments_dict = model_dict['segments_dict']
    documents_dict = model_dict['documents_dict']
    countries_dict = model_dict['countries_dict']
    encoded_segments = model_dict['encoded_segments']

    if hasattr(segments_dict, 'candidates') and not config.fuzzy_matching:
        scoped = segments_dict.subset(filtered_doc_ids)
        print(f'📊 Searching {len(scoped):,} segments in {len(filtered_doc_ids)} documents')
        _source = scoped.candidates(keyword)
    else:
        selected = set(filtered_doc_ids)
        relevant_segments = (sid for sid in encoded_segments if sid.split('/')[0] in selected)
        _source = ((sid, segments_dict[sid]) for sid in relevant_segments if sid in segments_dict)
    for segment_id, segment_data in tqdm(_source, desc=f'Searching "{keyword}"', leave=False):
        doc_id = segment_id.split('/')[0]
        segment_text = get_segment_text_safe(segment_data)

        if not segment_text:
            continue

        # Check for pattern match
        match_found = False
        match_score = 100.0  # Perfect match score

        if pattern.search(segment_text):
            match_found = True
        elif config.fuzzy_matching and FUZZY_AVAILABLE:
            # Fuzzy matching fallback
            fuzzy_score = fuzz.partial_ratio(keyword.lower(), segment_text.lower())
            if fuzzy_score >= config.fuzzy_threshold:
                match_found = True
                match_score = float(fuzzy_score)

        if match_found:
            # Get document metadata
            doc_data = documents_dict.get(doc_id, {})
            country_iso = doc_data.get('iso', '')

            # Get country name from countries_dict
            country_info = countries_dict.get(country_iso, {})
            country_name = country_info.get('country', doc_data.get('country', 'Unknown'))

            # Get context if requested
            context_before, context_after = get_context_window(
                segment_id, model_dict, config.context_window
            )

            result = SearchResult(
                segment_id=segment_id,
                text=segment_text,
                score=match_score,
                document_id=doc_id,
                document_name=doc_data.get('document', doc_data.get('name', '')),
                country=country_name,
                country_iso=country_iso,
                year=int(doc_data.get('year', 0)) if str(doc_data.get('year', '')).isdigit() else 0,
                doc_type=doc_data.get('type', ''),
                context_before=context_before,
                context_after=context_after
            )

            # Enrich with metadata
            result = enrich_result_with_metadata(result)
            results.append(result)

            # Respect max results limit if set
            if config.max_results_per_topic is not None and len(results) >= config.max_results_per_topic:
                print(f'⚠️  Reached maximum results limit ({config.max_results_per_topic})')
                break

    results, _merged = _count_once(results, config)
    print(f'✓ Found {len(results)} matches' + (f' ({_merged} original-language duplicates of an English match merged)' if _merged else ''))

    # Perform clustering if we have multiple results
    cluster_info_dict = {}
    if len(results) > 1:
        try:
            cluster_info_dict = cluster_search_results_internal(
                results, model_dict, config.cluster_threshold, search_query=keyword
            )
        except Exception as e:
            print(f'⚠️  Clustering error: {str(e)}')
            # Fallback: all singletons
            cluster_info_dict = {
                'cluster_dict': {'singletons': list(range(len(results)))},
                'num_clusters': 0,
                'num_singletons': len(results),
                'cluster_names': {'singletons': 'Unclustered'}
            }
            for result in results:
                result.cluster_id = 'singletons'
                result.cluster_name = 'Unclustered'
    elif len(results) == 1:
        # Single result handling
        cluster_info_dict = {
            'cluster_dict': {'singletons': [0]},
            'num_clusters': 0,
            'num_singletons': 1,
            'cluster_names': {'singletons': 'Unclustered'}
        }
        results[0].cluster_id = 'singletons'
        results[0].cluster_name = 'Unclustered'

    cluster_info_dict['pairs_merged'] = _merged
    return results, cluster_info_dict

# Batch search function
def run_metadata_batch_search(config: SearchConfig, model_dict, encoder):
    """Run batch search with metadata filtering (supports both semantic and keyword search)."""
    from datetime import datetime

    start_time = datetime.now()
    all_results = {}
    all_cluster_info = {}
    total_results = 0

    search_mode = 'Boolean' if config.boolean_mode else ('Keyword' if config.keyword_search else 'Semantic')
    print(f'\n🚀 Starting batch {search_mode.lower()} search...')
    print(f'📝 Topics: {len(config.topics)}')
    print(f'🔍 Filters: {config.filters.get_summary()}')
    print()

    for topic in tqdm(config.topics, desc='Searching topics'):
        try:
            if config.boolean_mode:
                # Boolean search mode
                results, cluster_info = run_boolean_search(topic, config, model_dict)
            elif config.keyword_search:
                # Keyword search mode
                results, cluster_info = run_keyword_search(topic, config, model_dict)
            else:
                # Semantic search mode
                results, cluster_info = run_similarity_search(topic, config, model_dict, encoder)

            all_results[topic] = results
            all_cluster_info[topic] = cluster_info
            total_results += len(results)

            if results:
                _enc, _oc = _en_orig_counts(results)
                _bd = f' [{_enc} English + {_oc} original-language]' if _oc else ''
                print(f'  ✓ {topic}: {len(results)} results{_bd} ({cluster_info.get("num_clusters", 0)} clusters, {cluster_info.get("num_singletons", 0)} singletons)')
            else:
                print(f'  ⚠️  {topic}: No results')
        except Exception as e:
            print(f'  ❌ {topic}: Error - {e}')
            import traceback
            traceback.print_exc()
            all_results[topic] = []
            all_cluster_info[topic] = {}

    end_time = datetime.now()
    execution_time = (end_time - start_time).total_seconds()

    print(f'\n✅ Search completed!')
    print(f'⏱️  Execution time: {execution_time:.2f} seconds')
    _all_r = [r for v in all_results.values() for r in v]
    _enc, _oc = _en_orig_counts(_all_r)
    if _oc:
        print(f'📊 Total results: {total_results}  ({_enc} English + {_oc} original-language; counted separately, not merged)')
    else:
        print(f'📊 Total results: {total_results}')

    return all_results, all_cluster_info

# Export function
def get_segment_viewer_uri(segment_id: str) -> str:
    """Absolute HTTP URL for CSV export, deep-linked to the matched segment."""
    try:
        document_id, segment_number = str(segment_id).split('/', 1)
    except ValueError:
        return ''
    base_url = os.environ.get('NSDDD_VOILA_BASE_URL', 'http://127.0.0.1:8867')
    return f'{base_url}/voila/files/XHR/{document_id}.html?segment={segment_number}'

def open_file(path):
    """Open a file with the OS default application (cross-platform)."""
    import platform
    p = str(path)
    system = platform.system()
    if system == 'Darwin':
        import subprocess
        subprocess.run(['open', p])
    elif system == 'Windows':
        os.startfile(p)
    else:
        import subprocess
        subprocess.run(['xdg-open', p])


def _eligible_country_stats(filters):
    """Per-country document count, year range and average length over the documents that PASS this search's filters
    (years, document type, membership and the rest). Coverage % and Hits / 1kw divide by these, not by every document
    the corpus holds for the country, so a country that joined an organisation part-way through the range is not
    penalised for the years before it joined. Returns None if it cannot be worked out (the corpus-wide figures are used)."""
    try:
        df = METADATA['documents']
        ids = set(apply_metadata_filters(df['File prefix'].astype(str).tolist(), filters))
        by_country = {}
        for prefix, country, year in zip(df['File prefix'], df['Country'], df['Year']):
            if str(prefix) in ids:
                by_country.setdefault(country, []).append((str(prefix), int(year)))
        out = {}
        for country, docs in by_country.items():
            wc = [DOC_NORM_STATS.get(d, {}).get('word_count', 0) for d, _ in docs]
            wc = [w for w in wc if w]
            years = [y for _, y in docs]
            out[country] = {'doc_count': len(docs), 'year_min': min(years), 'year_max': max(years),
                            'avg_words': int(round(sum(wc) / len(wc))) if wc else 0}
        return out
    except Exception:
        return None


def _compute_norm_context_rows(results, eligible=None):
    """Return a list of data dicts for the normalisation context CSV.

    Each dict contains per-country stats for the given list of SearchResults.
    Returns an empty list if COUNTRY_NORM_STATS is unavailable or no matches found.
    """
    if not COUNTRY_NORM_STATS:
        return []

    from collections import defaultdict
    cstats = defaultdict(lambda: {'segs': 0, 'docs': set(), 'scores': [], 'doc_years': {}})
    for r in results:
        if r.country in COUNTRY_NORM_STATS:
            cstats[r.country]['segs'] += 1
            cstats[r.country]['docs'].add(r.document_id)
            cstats[r.country]['scores'].append(r.score)
            if r.document_id and r.year:
                cstats[r.country]['doc_years'][r.document_id] = r.year

    rows = []
    for country in sorted(cstats.keys(), key=lambda c: -cstats[c]['segs']):
        ns = eligible[country] if eligible and country in eligible else COUNTRY_NORM_STATS[country]
        cs = cstats[country]
        hits = cs['segs']
        matched_docs = len(cs['docs'])
        total_docs = ns['doc_count']
        coverage = matched_docs / total_docs * 100 if total_docs else 0
        mean_score = sum(cs['scores']) / len(cs['scores']) if cs['scores'] else 0
        # avg_words from matched documents only
        matched_wcs = [DOC_NORM_STATS[d]['word_count'] for d in cs['docs']
                       if d in DOC_NORM_STATS and DOC_NORM_STATS[d]['word_count']]
        avg_words = int(round(sum(matched_wcs) / len(matched_wcs))) if matched_wcs else (ns['avg_words'] or 0)
        total_words_k = avg_words * total_docs / 1000
        hits_per_1k = hits / total_words_k if total_words_k else 0
        # year range from matched documents only
        valid_years = list(cs['doc_years'].values())
        year_min = min(valid_years) if valid_years else ns['year_min']
        year_max = max(valid_years) if valid_years else ns['year_max']
        rows.append({
            'country': country,
            'hits': hits,
            'matched_docs': matched_docs,
            'coverage_pct': round(coverage, 2),
            'mean_score': round(mean_score, 4),
            'hits_per_1k': round(hits_per_1k, 4),
            'total_docs': total_docs,
            'year_min': year_min,
            'year_max': year_max,
            'avg_words': avg_words,
        })
    return rows

def _cluster_headings_note(cluster_info):
    """One line saying where the cluster headings came from (so nobody has to guess)."""
    src = (cluster_info or {}).get('label_source', 'numbers')
    n = (cluster_info or {}).get('num_clusters', 0)
    if src == 'ai':
        return 'AI-written by a small local model (Qwen3-0.6B); check each against the example sentences'
    if src == 'ai_partial':
        k = len((cluster_info or {}).get('ai_named', []))
        return (f'AI-written by a small local model for {k} of {n} clusters; the others keep their numbers. '
                'Check each heading against the example sentences')
    return 'Numbered, largest cluster first (no AI headings)'


def export_search_results(all_results: Dict[str, List[SearchResult]], all_cluster_info: Dict,
                         config: SearchConfig, output_dir: str = 'outputs', enable_export: bool = True,
                         iterative_names: bool = False, file_stem: Optional[str] = None,
                         timestamp: Optional[str] = None):
    """Export search results with metadata, context and cluster headings.

    config.export_format 'xlsx' (default): one Excel workbook with the sheets Results, Search settings and
    Normalisation context. Links are real hyperlinks, so they work in every language version of Excel.
    config.export_format 'csv': a results CSV (settings block, then the table) and a normalisation-context CSV, with
    plain web addresses in the Link column. A CSV is split on commas, which Excel under continental European regional
    settings does not expect: it puts everything in column A (use Data > From Text/CSV and choose the comma).
    file_stem names the file when given; otherwise the topic does. timestamp fixes the iterative-name suffix, so that
    exporting again after the AI headings are added overwrites the first file instead of leaving a second.
    """
    if not enable_export:
        return

    fmt = str(getattr(config, 'export_format', 'xlsx') or 'xlsx').lower()
    if fmt == 'xlsx':
        try:
            from _library.xlsx_writer import write_xlsx, Link
        except Exception as e:
            print(f'⚠️  Could not load the Excel writer ({e}); writing CSV files instead.')
            fmt = 'csv'

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    print(f'📁 Export directory: {output_path}')

    search_type = 'Boolean Search' if config.boolean_mode else ('Keyword' if config.keyword_search else 'Semantic')

    written_files = []
    _eligible = _eligible_country_stats(config.filters)
    for topic, results in all_results.items():
        if not results:
            continue

        cluster_info = all_cluster_info.get(topic, {})
        num_clusters = cluster_info.get('num_clusters', 0)
        num_singletons = cluster_info.get('num_singletons', len(results))

        # Create safe filename with keyword suffix if applicable
        safe_topic = ''.join(c if c.isalnum() or c in (' ', '_') else '_' for c in (file_stem or topic)).strip().replace(' ', '_')[:50]
        safe_topic = safe_topic.lstrip('_')
        safe_topic = re.sub(r'_+', '_', safe_topic)
        safe_topic = safe_topic.rstrip('_')

        stem = f'{safe_topic}_keyword' if config.keyword_search else safe_topic
        if iterative_names:
            stem = f'{stem}_{timestamp or datetime.now().strftime("%Y%m%d_%H%M%S")}'
        ext = 'xlsx' if fmt == 'xlsx' else 'csv'
        filename = f'{stem}.{ext}'
        norm_filename = f'{stem}_norm_context.csv'
        filepath = output_path / filename
        norm_filepath = output_path / norm_filename

        search_threshold_display = 'n/a' if config.keyword_search else round(config.search_threshold, 3)
        cluster_threshold_display = round(config.cluster_threshold, 3)
        settings = [
            ['Topic', topic],
            ['Search type', search_type],
            ['Search threshold', search_threshold_display],
            ['Cluster threshold', cluster_threshold_display],
            ['Number of search results', len(results)],
            ['Number of clusters', num_clusters],
            ['Unclustered results', num_singletons],
            ['Cluster headings', _cluster_headings_note(cluster_info)],
            ['Context window', config.context_window],
            ['Active filters', config.filters.get_summary()],
            ['Links', 'Open only while NSDDD is running on the computer that made this file'],
            ['Clustering method', cluster_info.get('clustering_note') or ('Ward linkage' if num_clusters else 'Unclustered')],
            ['Passage counting', ('Each passage counted once: ' + str(cluster_info.get('pairs_merged', 0)) + ' original-language matches merged into the English match of the same passage'
                                  if getattr(config, 'count_passages_once', True) else 'Original-language and English matches of one passage counted separately')],
        ]
        columns = ['Cluster ID', 'Cluster Name', 'Search score', 'Segment text', 'Context before', 'Context after', 'Segment ID',
                   'Document name', 'Document year', 'Country', 'Country ISO', 'Region', 'Subregion',
                   'Income Group', 'Freedom Status', 'Link', 'Matched language', 'Original language text', 'Aligned segment ID(s)',
                   'Document languages', 'Cluster example']
        widths = [12, 30, 9, 70, 40, 40, 12, 36, 8, 16, 8, 14, 18, 14, 14, 18, 18, 70, 18, 30, 50]

        table = []
        for result in results:
            cluster_id = result.cluster_id if result.cluster_id is not None else 'singletons'
            cluster_name = result.cluster_name if result.cluster_name else str(cluster_id)
            # 'Link' opens the parallel page when the document has two languages, else the segment's own page.
            # 'Segment text' is always English and 'Original language text' always the original: the matched
            # segment on its side and its confidently aligned counterpart (>= MIN_ALIGN_SIM) on the other.
            _p = str(result.segment_id).split('/')[0]
            _is_orig = '/o/' in result.segment_id
            _L = segment_links(result.segment_id)
            _label = 'Open parallel text' if _L.get('parallel') else 'Open segment'
            link = Link(_L['main'], _label) if fmt == 'xlsx' else _L['main']
            _lang_disp = _matched_language(result.segment_id)
            _other_txt, _o_label, _o_lang, _o_flag, _other_ids = _aligned_other(result.segment_id)
            _seg_en = _other_txt if _is_orig else result.text
            _orig_txt = result.text if _is_orig else _other_txt
            _aligned_seg_ids = ', '.join(_seg_nums(_other_ids))
            table.append([
                str(cluster_id),
                cluster_name,
                float(f'{result.score:.4f}') if fmt == 'xlsx' else f'{result.score:.4f}',
                _seg_en,
                result.context_before,
                result.context_after,
                result.segment_id,
                result.document_name,
                result.year,
                result.country,
                result.country_iso,
                result.region,
                result.subregion,
                result.income_group,
                result.freedom_status,
                link,
                _lang_disp,
                _orig_txt,
                _aligned_seg_ids,
                '; '.join(_doc_languages(_p)),
                getattr(result, 'cluster_example', '')
            ])

        norm_rows = _compute_norm_context_rows(results, eligible=_eligible)
        norm_notes = [
            ['Normalisation context: data computed from documents cited in these search results only.'],
            ['Years covered and Avg doc length reflect matched documents (cited in these results) only.'],
            ['Coverage % = matched docs / documents for that country that pass this search\'s filters (years, document type, membership and so on).'],
            ['Hits / 1kw = hits / (avg matched doc length × eligible docs / 1,000), where eligible docs are those that pass the filters.'],
            ['Mean score = average cosine similarity of matched segments.'],
        ]
        norm_columns = ['Country', 'Hits', 'Matched docs', 'Coverage %', 'Mean score',
                        'Hits / 1kw', 'Docs eligible', 'Year min', 'Year max', 'Avg doc length (words)']
        norm_table = [[r['country'], r['hits'], r['matched_docs'], r['coverage_pct'], r['mean_score'], r['hits_per_1k'],
                       r['total_docs'], r['year_min'], r['year_max'], r['avg_words']] for r in norm_rows]

        if fmt == 'xlsx':
            sheets = [{'name': 'Results', 'rows': [columns] + table, 'header_row': 0, 'widths': widths},
                      {'name': 'Search settings', 'rows': settings, 'header_row': None, 'widths': [28, 90]}]
            if norm_table:
                sheets.append({'name': 'Normalisation context',
                               'rows': [settings[0], settings[1], settings[9]] + [['']] + norm_notes + [['']] + [norm_columns] + norm_table,
                               'header_row': 3 + 1 + len(norm_notes) + 1, 'widths': [28, 10, 14, 12, 12, 12, 16, 10, 10, 22]})
            write_xlsx(filepath, sheets)
            print(f'  ✓ Exported: {filename} ({len(results)} results, {num_clusters} clusters + {num_singletons} unclustered; '
                  f'sheets: Results, Search settings' + (', Normalisation context)' if norm_table else ')'))
            written_files.append(filepath)
            if norm_table:                      # also as its own CSV (with its download button), as before the Excel export
                with open(norm_filepath, 'w', newline='', encoding='utf-8-sig') as f:
                    f.write('sep=,\r\n')       # tells Excel the separator whatever the regional settings (Danish Excel expects ';')
                    csv.writer(f).writerows(settings[:5] + [settings[9]] + [['']] + norm_notes + [['']] + [norm_columns] + norm_table)
                print(f'  ✓ Exported: {norm_filename}')
                written_files.append(norm_filepath)
        else:
            with open(filepath, 'w', newline='', encoding='utf-8-sig') as f:
                f.write('sep=,\r\n')           # tells Excel the separator whatever the regional settings (Danish Excel expects ';')
                csv.writer(f).writerows(settings + [['']] + [columns] + table)
            print(f'  ✓ Exported: {filename} ({len(results)} results, {num_clusters} clusters + {num_singletons} unclustered)')
            written_files.append(filepath)
            if norm_table:
                with open(norm_filepath, 'w', newline='', encoding='utf-8-sig') as f:
                    f.write('sep=,\r\n')
                    csv.writer(f).writerows(settings[:5] + [settings[9]] + [['']] + norm_notes + [['']] + [norm_columns] + norm_table)
                written_files.append(norm_filepath)
                print(f'  ✓ Exported: {norm_filename} ({len(norm_rows)} countries, normalisation context)')

    return written_files

# Display function
# --- v4.0 multilingual result rendering (language badge, original<->English coupling, dedup, filter) ---
LANG_VIEW = 'all'  # 'all' | 'en' | 'orig'
LANG_LABEL = {'en':'English','es':'Spanish','pt':'Portuguese','fr':'French','de':'German','it':'Italian','nl':'Dutch','no':'Norwegian','da':'Danish','sv':'Swedish','fi':'Finnish','ru':'Russian','uk':'Ukrainian','cs':'Czech','sk':'Slovak','sl':'Slovenian','hr':'Croatian','bg':'Bulgarian','mk':'Macedonian','pl':'Polish','ro':'Romanian','hu':'Hungarian','sq':'Albanian','lv':'Latvian','lt':'Lithuanian','el':'Greek','ka':'Georgian','zh':'Chinese','ko':'Korean','ja':'Japanese','mn':'Mongolian','ar':'Arabic','id':'Indonesian','ky':'Kyrgyz','bs':'Bosnian','is':'Icelandic','th':'Thai','ms':'Malay','uz':'Uzbek','und':'Original'}

def _doc_languages(prefix):
    """The document's languages as the metadata records them, e.g. ['English (Translated)', 'Norwegian (Original)']."""
    return list((MODEL_DICT.get('documents_dict', {}).get(str(prefix), {}) or {}).get('languages') or [])

def _orig_lang_name(prefix, code=''):
    """Name of the document's original language from the metadata (first non-English language), else from the code."""
    for l in _doc_languages(prefix)[1:]:
        n = re.sub(r'\s*\(original\)\s*$', '', l, flags=re.I).strip()
        if n and 'english' not in n.lower():
            return n
    return LANG_LABEL.get(code, code.upper() if code else 'Original')

def _english_label(prefix):
    langs = _doc_languages(prefix)
    return 'English (translated)' if langs and 'translated' in langs[0].lower() else 'English (official)'

def _matched_language(seg_id):
    p = str(seg_id).split('/')[0]
    return (_orig_lang_name(p, _seg_lang(seg_id)) + ' (original)') if '/o/' in str(seg_id) else _english_label(p)

def _en_orig_counts(results):
    o = sum(1 for r in results if '/o/' in getattr(r, 'segment_id', ''))
    return len(results) - o, o


def _seg_lang(seg_id):
    try:
        return (MODEL_DICT.get('segments_dict', {}).get(seg_id, {}) or {}).get('lang', 'en')
    except Exception:
        return 'en'

def _lang_badge(lang, prefix=None):
    if lang == 'en':
        return '<span style="display:inline-block;font-size:10px;font-weight:700;background:#e7f0ff;color:#1f6feb;border-radius:20px;padding:1px 8px;margin-left:6px;">EN &middot; English</span>'
    label = _orig_lang_name(prefix, lang) if prefix else LANG_LABEL.get(lang, lang.upper())
    return ('<span style="display:inline-block;font-size:10px;font-weight:700;background:#e9f7ef;color:#0b8457;border-radius:20px;padding:1px 8px;margin-left:6px;">' + escape(lang.upper()) + ' &middot; ' + escape(label) + ' (original)</span>')

def _aligned_english_text(seg_id):
    if '/o/' not in seg_id:
        return ''
    links = MODEL_DICT.get('alignment_dict', {}).get(seg_id.split('/')[0])
    if not links:
        return ''
    segs = MODEL_DICT.get('segments_dict', {})
    for orig_ids, en_ids, sim in links:
        if seg_id in orig_ids:
            return ' '.join(t for t in ((segs.get(e, {}) or {}).get('text', '') for e in en_ids) if t)
    return ''

MIN_ALIGN_SIM = 0.70   # calibrated 2026-09-29 (corpus_alignment_calibrate.py, free Qwen judging 480 sampled links):
                       # >= 0.70: 95% correct or partly correct; 0.45-0.70: most pair the wrong sentences

MIN_DOC_ANCHOR = 0.0    # document-level gate retired 2026-09-29: the anchor score (shared numbers) was low for
                        # well-aligned documents (e.g. 433, 429), and the per-link threshold above is what matters


def _aligned_other(seg_id):
    """(text, label, lang, flag, other_ids). flag: '' none; 'low' a link exists but
    sim < MIN_ALIGN_SIM; 'doc' the document alignment is unreliable (anchor < MIN_DOC_ANCHOR).
    other_ids = the counterpart (other-language) segment id(s), or [] when none/flagged.
    When flagged, no (potentially wrong) counterpart text is returned."""
    parts = str(seg_id).split('/')
    if MODEL_DICT.get('alignment_anchor', {}).get(parts[0], 1.0) < MIN_DOC_ANCHOR:
        return '', '', '', 'doc', []
    links = MODEL_DICT.get('alignment_dict', {}).get(parts[0])
    if not links:
        return '', '', '', '', []
    segs = MODEL_DICT.get('segments_dict', {})
    is_orig = (len(parts) == 3 and parts[1] == 'o')
    for orig_ids, en_ids, sim in links:
        mine = orig_ids if is_orig else en_ids
        if seg_id in mine:
            if (sim or 0) < MIN_ALIGN_SIM:
                return '', '', '', 'low', []
            other = list(en_ids if is_orig else orig_ids)
            txt = ' '.join((segs.get(o, {}) or {}).get('text', '') for o in other).strip()
            if not txt:
                return '', '', '', '', []
            if is_orig:
                return txt, _english_label(parts[0]), 'en', '', other
            lang = (segs.get(other[0], {}) or {}).get('lang', '') if other else ''
            return txt, _orig_lang_name(parts[0], lang) + ' (original)', lang, '', other
    return '', '', '', '', []


def _seg_nums(ids):
    """Bare 1-based segment numbers from seg ids ('380/o/85'->'85', '712/1290'->'1290')."""
    out = []
    for s in ids:
        p = str(s).split('/')
        out.append(p[2] if (len(p) == 3 and p[1] == 'o') else (p[1] if len(p) >= 2 else str(s)))
    return out


def _collapse_pairs(results):
    """DISPLAY-only: when both an original-language segment and its aligned English are present,
    drop the ENGLISH card and keep the ORIGINAL (its badge + original text lead, with the
    English translation beneath). The result COUNT is computed elsewhere and is NOT affected."""
    align = MODEL_DICT.get('alignment_dict', {})
    present = {r.segment_id: r for r in results}
    skip = set()
    for r in results:
        sid = r.segment_id
        if sid in skip:
            continue
        parts = sid.split('/')
        links = align.get(parts[0])
        if not links:
            continue
        if len(parts) != 2:          # only an English id ("{p}/{j}") can be dropped for its original
            continue
        for o_ids, e_ids, sim in links:
            if sid in e_ids:
                if (sim or 0) >= MIN_ALIGN_SIM and any(o in present for o in o_ids):
                    skip.add(sid)
                break
    return [r for r in results if r.segment_id not in skip]


def dedupe_pair_matches(results, keep='en'):
    """Count each passage once: when a segment and its confidently aligned counterpart (>= MIN_ALIGN_SIM) both matched,
    keep one result. keep='en' keeps the English match (its row carries the original text as well, so nothing is lost);
    keep='orig' keeps the original-language match. A match whose counterpart did not match is kept either way.
    Returns (results, number_dropped). Runs before clustering, so counts, clusters, charts and exports agree."""
    align = MODEL_DICT.get('alignment_dict', {}) or {}
    present = {r.segment_id for r in results}
    drop, index = set(), {}
    for r in results:
        sid = r.segment_id
        parts = sid.split('/')
        is_orig = len(parts) == 3 and parts[1] == 'o'
        if is_orig != (keep == 'en'):           # only the other language's match can be dropped
            continue
        links = align.get(parts[0])
        if not links:
            continue
        if parts[0] not in index:
            idx = {}
            for o_ids, e_ids, sim in links:
                mine, other = (o_ids, e_ids) if keep == 'en' else (e_ids, o_ids)
                for s in mine:
                    idx.setdefault(s, []).append((other, sim))
            index[parts[0]] = idx
        for other, sim in index[parts[0]].get(sid, ()):
            if (sim or 0) >= MIN_ALIGN_SIM and any(o in present for o in other):
                drop.add(sid)
                break
    if not drop:
        return results, 0
    return [r for r in results if r.segment_id not in drop], len(drop)


def _count_once(results, config, cluster_view=None):
    """Apply dedupe_pair_matches when the search asks for it (the default)."""
    if not getattr(config, 'count_passages_once', True):
        return results, 0
    return dedupe_pair_matches(results, 'orig' if LANG_VIEW == 'orig' else 'en')


def _coupled_en_ids_for(results):
    skip = set()
    align = MODEL_DICT.get('alignment_dict', {})
    for r in results:
        sid = r.segment_id
        if '/o/' in sid:
            for orig_ids, en_ids, sim in align.get(sid.split('/')[0], []) or []:
                if sid in orig_ids:
                    skip.update(en_ids)
    return skip

def get_bilingual_viewer_url(seg_id):
    parts = str(seg_id).split('/')
    p = parts[0]
    if p not in MODEL_DICT.get('alignment_dict', {}):
        return ''
    if len(parts) == 3 and parts[1] == 'o':
        anchor = 'o' + parts[2]
    elif len(parts) == 2:
        anchor = 'e' + parts[1]
    else:
        return ''
    return f"/voila/files/XHR/{p}_bilingual.html?segment={anchor}"


def _voila_base():
    return os.environ.get('NSDDD_VOILA_BASE_URL', 'http://127.0.0.1:8867').rstrip('/')

def segment_links(seg_id, absolute=True):
    """All links for a matched segment, shared by the results panel and the CSV (2026-09-29).
    english / original: that version's page, opened at this segment's number in it (for the other version, the
    confidently aligned counterpart's number; the top of the page when there is none).
    parallel: the side-by-side page at this segment, for documents with two languages and an alignment.
    main: parallel when there is one, else the matched segment's own page."""
    parts = str(seg_id).split('/')
    p = parts[0]
    is_orig = len(parts) == 3 and parts[1] == 'o'
    own = parts[2] if is_orig else (parts[1] if len(parts) >= 2 else '')
    other = _aligned_other(seg_id)[4]
    other_num = _seg_nums(other)[0] if other else ''
    base = (_voila_base() if absolute else '') + '/voila/files/XHR/'
    q = lambda n: ('?segment=' + n) if n else ''
    links = {'english': base + p + '.html' + q(other_num if is_orig else own)}
    if _has_original(p):
        links['original'] = base + p + '_original.html' + q(own if is_orig else other_num)
    if p in MODEL_DICT.get('alignment_dict', {}):
        links['parallel'] = base + p + '_bilingual.html?segment=' + ('o' if is_orig else 'e') + own
    links['main'] = links.get('parallel') or (links.get('original') if is_orig else links['english'])
    return links

def segment_links_html(seg_id):
    """'Parallel · English · Original' link row for a result card."""
    L = segment_links(seg_id, absolute=False)
    a = lambda u, t: ('<a href="' + escape(u) + '" target="_blank" rel="noopener noreferrer" '
                      'style="color:#1565c0;text-decoration:none;">' + t + '</a>')
    items = []
    if L.get('parallel'):
        items.append(a(L['parallel'], '⇋ Parallel text'))
    items.append(a(L['english'], 'English'))
    if L.get('original'):
        items.append(a(L['original'], escape(_orig_lang_name(str(seg_id).split('/')[0], _seg_lang(seg_id))) + ' (original)'))
    return ('<p style="margin:4px 0 0 0;color:#607d8b;font-size:11px;">🔗 ' + ' &nbsp;·&nbsp; '.join(items) +
            ' &nbsp;|&nbsp; Segment ' + escape(str(seg_id)) + '</p>')

def _has_original(prefix):
    return (str(prefix) + '/o/1') in MODEL_DICT.get('segments_dict', {})

def get_doc_viewer_uri(prefix, original=False):
    base = os.environ.get('NSDDD_VOILA_BASE_URL', 'http://127.0.0.1:8867').rstrip('/')
    return base + '/voila/files/XHR/' + str(prefix) + ('_original' if original else '') + '.html'

def get_doc_viewer_link_html(prefix, original, label):
    return ('<a href="' + escape(get_doc_viewer_uri(prefix, original)) +
            '" target="_blank" rel="noopener noreferrer" style="color:#1565c0;text-decoration:none;">' +
            escape(label) + '</a>')


def display_batch_results(all_results: Dict[str, List[SearchResult]], all_cluster_info: Dict,
                         config: SearchConfig, max_per_topic: int = 5):
    """Display batch search results with cluster names and optional context."""
    for topic, results in all_results.items():
        topic_safe = escape(topic)
        if not results:
            display(HTML(f'''
            <div style="background-color: #ffebee; padding: 15px; margin: 20px 0;
                        border-radius: 8px; border-left: 4px solid #f44336;">
                <strong>❌ No results found for: "{topic_safe}"</strong>
            </div>
            '''))
            continue

        cluster_info = all_cluster_info.get(topic, {})
        num_clusters = cluster_info.get('num_clusters', 0)
        num_singletons = cluster_info.get('num_singletons', 0)

        # Header
        search_type = 'Boolean Search' if config.boolean_mode else ('Keyword Search' if config.keyword_search else 'Semantic Search')
        display(HTML(f'''
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    padding: 20px; border-radius: 10px; color: white; margin: 20px 0;">
            <h2 style="margin: 0; font-size: 24px;">🔍 {escape(search_type)} Results: "{topic_safe}"</h2>
            <p style="margin: 10px 0 0 0; opacity: 0.9;">
                Found {len(results)} results |
                {num_clusters} clusters + {num_singletons} singletons |
                {len({r.country_iso for r in results if r.country_iso})} {'country' if len({r.country_iso for r in results if r.country_iso}) == 1 else 'countries'} |
                {len({r.document_id for r in results if r.document_id})} {'document' if len({r.document_id for r in results if r.document_id}) == 1 else 'documents'}
            </p>
        </div>
        '''))

        # Group by document
        by_document = defaultdict(list)
        for result in results[:max_per_topic * 5]:  # Show more results
            by_document[result.document_id].append(result)

        # Display each document group
        display(HTML('<p style="margin:12px 0 4px 0;color:#888;font-size:12px;font-style:italic;">Sample results</p>'))
        for doc_id, doc_results in list(by_document.items())[:max_per_topic]:
            first_result = doc_results[0]

            # Document header with metadata
            metadata_tags = []
            if first_result.region:
                metadata_tags.append(f'🌍 {escape(first_result.region)}')
            if first_result.subregion:
                metadata_tags.append(f'📍 {escape(first_result.subregion)}')
            if first_result.income_group:
                metadata_tags.append(f'💰 {escape(first_result.income_group)}')
            if first_result.freedom_status:
                metadata_tags.append(f'🗳️ {escape(first_result.freedom_status)}')

            metadata_str = ' | '.join(metadata_tags)

            display(HTML(f'''
            <div style="background-color: #f5f5f5; padding: 15px; margin: 15px 0;
                        border-radius: 8px; border-left: 5px solid #4CAF50;">
                <h3 style="margin: 0 0 10px 0; color: #546e7a;">
                    📄 {escape(first_result.country)} ({first_result.year})
                </h3>
                <p style="margin: 5px 0; color: #555; font-size: 14px;">
                    <strong>{escape(first_result.document_name)}</strong>
                </p>
                <p style="margin: 5px 0; color: #777; font-size: 12px;">
                    {metadata_str}
                </p>
                <p style="margin: 5px 0; color: #999; font-size: 12px;">
                    📊 {len(doc_results)} results in this document
                </p>
            </div>
            '''))

            # Display individual results
            for i, result in enumerate(doc_results[:3], 1):
                score_indicator = '🎯' if result.score > 0.8 else '✅' if result.score > 0.65 else '📍'

                # Display cluster name if available, otherwise cluster ID
                if result.cluster_name and result.cluster_name != 'Unclustered':
                    cluster_display = f' | 🏷️ {escape(result.cluster_name)}'
                elif result.cluster_id is not None and result.cluster_id != 'singletons':
                    cluster_display = f' | Cluster {escape(str(result.cluster_id))}'
                else:
                    cluster_display = ''

                # Build context HTML if context is available
                context_html = ''
                if result.context_before or result.context_after:
                    context_html = '<div style="background-color: #f9f9f9; padding: 10px; margin: 8px 0; border-left: 3px solid #ccc; font-size: 12px; color: #666; font-style: italic;">'
                    if result.context_before:
                        cb = escape(result.context_before[:300]) + ('...' if len(result.context_before) > 300 else '')
                        context_html += f'<p style="margin: 0 0 5px 0;"><strong>↑ Before:</strong> {cb}</p>'
                    if result.context_after:
                        ca = escape(result.context_after[:300]) + ('...' if len(result.context_after) > 300 else '')
                        context_html += f'<p style="margin: 5px 0 0 0;"><strong>↓ After:</strong> {ca}</p>'
                    context_html += '</div>'

                result_text = escape(result.text[:500]) + ('...' if len(result.text) > 500 else '')
                _main = segment_links(result.segment_id, absolute=False)['main']
                result_body = f'<a href="{escape(_main)}" target="_blank" rel="noopener noreferrer" style="color:#1565c0;text-decoration:none;">{result_text}</a>'
                result_link_html = segment_links_html(result.segment_id)
                display(HTML(f'''
                <div style="background-color: white; padding: 12px; margin: 8px 0 8px 20px;
                            border-radius: 5px; border-left: 3px solid #9C27B0;">
                    <p style="margin: 0 0 5px 0; color: #888; font-size: 11px;">
                        {score_indicator} Score: {result.score:.3f}{cluster_display}
                    </p>
                    <p style="margin: 0 0 8px 0; color: #333; line-height: 1.6;">
                        {result_body}
                    </p>
                    {result_link_html}
                    {context_html}
                </div>
                '''))

            if len(doc_results) > 3:
                display(HTML(f'''
                <p style="margin: 5px 0 5px 20px; color: #999; font-size: 12px; font-style: italic;">
                    ... and {len(doc_results) - 3} more results in this document
                </p>
                '''))



def _build_norm_panel_html(all_results, eligible=None):
    """Return an HTML string for the normalisation context panel, or '' if unavailable."""
    if not COUNTRY_NORM_STATS:
        return ''

    from collections import defaultdict
    _cstats = defaultdict(lambda: {'segs': 0, 'docs': set(), 'scores': [], 'doc_years': {}})
    for results in all_results.values():
        for r in results:
            if r.country in COUNTRY_NORM_STATS:
                _cstats[r.country]['segs'] += 1
                _cstats[r.country]['docs'].add(r.document_id)
                _cstats[r.country]['scores'].append(r.score)
                if r.document_id and r.year:
                    _cstats[r.country]['doc_years'][r.document_id] = r.year

    _panel_countries = sorted(_cstats.keys(), key=lambda c: -_cstats[c]['segs'])
    if not _panel_countries:
        return ''

    def _row(c):
        ns = eligible[c] if eligible and c in eligible else COUNTRY_NORM_STATS[c]
        cs = _cstats[c]
        hits = cs['segs']
        matched_docs = len(cs['docs'])
        total_docs = ns['doc_count']
        coverage = matched_docs / total_docs * 100 if total_docs else 0
        mean_score = sum(cs['scores']) / len(cs['scores']) if cs['scores'] else 0
        # avg_words from matched documents only
        matched_wcs = [DOC_NORM_STATS[d]['word_count'] for d in cs['docs']
                       if d in DOC_NORM_STATS and DOC_NORM_STATS[d]['word_count']]
        avg_words = int(round(sum(matched_wcs) / len(matched_wcs))) if matched_wcs else (ns['avg_words'] or 0)
        total_words_k = avg_words * total_docs / 1000
        hits_per_1k = hits / total_words_k if total_words_k else 0
        # year range from matched documents only
        valid_years = list(cs['doc_years'].values())
        year_min = min(valid_years) if valid_years else ns['year_min']
        year_max = max(valid_years) if valid_years else ns['year_max']
        return (
            f'<tr>'
            f'<td style="padding:4px 12px 4px 0;" data-val="{escape(c)}">{escape(c)}</td>'
            f'<td style="padding:4px 12px 4px 0;text-align:center;" data-val="{hits}">{hits}</td>'
            f'<td style="padding:4px 12px 4px 0;text-align:center;" data-val="{matched_docs}">{matched_docs}</td>'
            f'<td style="padding:4px 12px 4px 0;text-align:center;" data-val="{coverage:.4f}">{coverage:.0f}%</td>'
            f'<td style="padding:4px 12px 4px 0;text-align:center;" data-val="{mean_score:.6f}">{mean_score:.3f}</td>'
            f'<td style="padding:4px 12px 4px 0;text-align:center;" data-val="{hits_per_1k:.6f}">{hits_per_1k:.2f}</td>'
            f'<td style="padding:4px 12px 4px 0;text-align:center;" data-val="{total_docs}">{total_docs}</td>'
            f'<td style="padding:4px 12px 4px 0;text-align:center;" data-val="{year_min}">{year_min}\u2013{year_max}</td>'
            f'<td style="padding:4px 0;text-align:right;" data-val="{avg_words}">{avg_words:,}</td>'
            f'</tr>'
        )

    _rows_html = ''.join(_row(c) for c in _panel_countries)

    _sort_script = (
        '<script>(function(){'
        'var tbl=document.getElementById("nsddd-norm-tbl");'
        'if(!tbl)return;'
        'var asc={};'
        'tbl.querySelectorAll("thead th").forEach(function(th,col){'
        'th.style.cursor="pointer";'
        'th.title="Click to sort";'
        'th.addEventListener("click",function(){'
        'asc[col]=!asc[col];'
        'var tbody=tbl.querySelector("tbody");'
        'var rows=Array.from(tbody.querySelectorAll("tr"));'
        'rows.sort(function(a,b){'
        'var av=a.cells[col].dataset.val;'
        'var bv=b.cells[col].dataset.val;'
        'var an=parseFloat(av),bn=parseFloat(bv);'
        'if(!isNaN(an)&&!isNaN(bn))return asc[col]?an-bn:bn-an;'
        'if(av<bv)return asc[col]?-1:1;'
        'if(av>bv)return asc[col]?1:-1;'
        'return 0;'
        '});'
        'rows.forEach(function(r){tbody.appendChild(r);});'
        'tbl.querySelectorAll("thead th").forEach(function(h,c){'
        'var s=h.querySelector(".nsi");'
        'if(s)s.textContent=c===col?(asc[col]?" \u25b2":" \u25bc"):"";'
        '});'
        '});'
        '});'
        '})()</script>'
    )

    _th_style = 'padding:4px 12px 6px 0;color:#01579b;user-select:none;'
    return (
        '\n    <details open style="margin:16px 0;">\n'
        '      <summary style="cursor:pointer;padding:10px 14px;background:#e8f4f8;\n'
        '                      border-left:4px solid #0288d1;border-radius:6px;\n'
        '                      font-size:13px;color:#01579b;font-weight:600;list-style:none;">\n'
        '        &#9432; Normalisation context &mdash; matched documents in these results\n'
        '      </summary>\n'
        '      <div style="background:#f0f9fc;border-left:4px solid #0288d1;border-radius:0 0 6px 6px;\n'
        '                  padding:12px 14px;font-size:13px;color:#333;">\n'
        '        <p style="margin:0 0 10px 0;color:#555;line-height:1.5;">\n'
        '          Sorted by hits. All data are computed from matched documents cited in these results only.\n'
        '          Years covered and Avg doc length reflect matched documents only.\n'
        '          Coverage&#37; = matched docs &divide; eligible docs (documents for that country that pass this search&rsquo;s filters).\n'
        '          Hits/1kw = hits &divide; (avg matched doc length &times; eligible docs &divide; 1000).\n'
        '          Mean score is the average cosine similarity of matched segments.\n'
        '          Click any column heading to sort.\n'
        '        </p>\n'
        '        <div style="overflow-x:auto;">\n'
        '        <table id="nsddd-norm-tbl" style="border-collapse:collapse;width:auto;white-space:nowrap;">\n'
        '          <thead>\n'
        '            <tr style="border-bottom:1px solid #b0d4e3;">\n'
        f'              <th style="{_th_style}text-align:left;">Country<span class="nsi"></span></th>\n'
        f'              <th style="{_th_style}text-align:center;">Hits<span class="nsi"></span></th>\n'
        f'              <th style="{_th_style}text-align:center;">Matched docs<span class="nsi"></span></th>\n'
        f'              <th style="{_th_style}text-align:center;">Coverage %<span class="nsi"></span></th>\n'
        f'              <th style="{_th_style}text-align:center;">Mean score<span class="nsi"></span></th>\n'
        f'              <th style="{_th_style}text-align:center;">Hits / 1kw<span class="nsi"></span></th>\n'
        f'              <th style="{_th_style}text-align:center;">Docs eligible<span class="nsi"></span></th>\n'
        f'              <th style="{_th_style}text-align:center;">Years covered<span class="nsi"></span></th>\n'
        f'              <th style="{_th_style}text-align:right;">Avg doc length (words)<span class="nsi"></span></th>\n'
        '            </tr>\n'
        '          </thead>\n'
        f'          <tbody>{_rows_html}</tbody>\n'
        '        </table>\n'
        '        </div>\n'
        f'      </div>\n'
        f'    </details>\n'
        f'    {_sort_script}\n'
        '    '
    )


In [ ]:
%%capture
# Boolean search implementation
from _library.boolean_parser import parse_boolean_query, BooleanQueryParser
from _library.boolean_executor import execute_boolean_query, BooleanQueryExecutor

def run_keyword_search_internal(keyword: str, model_dict: dict) -> List[Tuple[str, float]]:
    """
    Internal keyword search function for Boolean executor.

    Performs basic keyword pattern matching without metadata filtering or clustering.
    Returns raw list of (segment_id, score) tuples.

    Args:
        keyword: Search term (may include wildcards * and ?)
        model_dict: Data model containing segments

    Returns:
        List of (segment_id, score) tuples
    """
    segments_dict = model_dict['segments_dict']

    # Escape special regex chars except * and ?
    import re
    pattern = re.escape(keyword)
    # Replace escaped wildcards with regex equivalents
    pattern = pattern.replace(r'\*', '.*').replace(r'\?', '.')
    # Add word boundaries
    pattern = r'\b' + pattern + r'\b'

    # Compile pattern (case-insensitive)
    try:
        regex = re.compile(pattern, re.IGNORECASE)
    except re.error as e:
        print(f'⚠️  Invalid regex pattern for term "{keyword}": {e}')
        return []

    # Search all segments
    results = []
    # Qwen3 app: candidates from the SQLite full-text index (same matches as a full scan, in milliseconds)
    _source = segments_dict.candidates(keyword) if hasattr(segments_dict, 'candidates') else segments_dict.items()
    for segment_id, segment_data in _source:
        text = get_segment_text_safe(segment_data)
        if not text:
            continue
        if regex.search(text):
            # Score is 100.0 for keyword matches (normalized to 0-100 range)
            results.append((segment_id, 100.0))

    return results


def run_boolean_search(query: str, config: SearchConfig, model_dict) -> Tuple[List[SearchResult], Dict]:
    """
    Run Boolean search with metadata filtering and clustering.

    Supports Boolean operators (AND, OR, NOT) with parentheses grouping.

    Args:
        query: Boolean query string (e.g., "cyber AND attack NOT terrorism")
        config: Search configuration
        model_dict: Data model containing segments and metadata

    Returns:
        Tuple of (results, cluster_info_dict)

    Example queries:
        - "cyber AND attack"
        - "(nuclear OR radiological) AND weapon"
        - "terrorism AND NOT domestic"
        - "(cyber* OR digital) AND threat AND NOT terrorism"
    """
    print(f'🔍 Boolean search: {query}')

    # Get filtered documents
    doc_df = METADATA['documents']
    all_doc_ids = doc_df['File prefix'].astype(str).tolist()
    filtered_doc_ids = apply_metadata_filters(all_doc_ids, config.filters)

    if not filtered_doc_ids:
        print('⚠️  No documents match the filter criteria')
        return [], {}

    print(f'📄 Searching {len(filtered_doc_ids)} documents')

    # Parse Boolean query into AST
    try:
        parser = BooleanQueryParser()
        ast = parser.parse(query)
        print(f'✓ Query parsed successfully')
    except Exception as e:
        print(f'❌ Query parse error: {e}')
        return [], {}

    # Create filtered model_dict with only selected documents
    segments_dict = model_dict['segments_dict']
    if hasattr(segments_dict, 'subset'):             # Qwen3 app: a view on the SQLite store, not a copy
        filtered_segments_dict = segments_dict.subset(filtered_doc_ids)
    else:
        _docs = set(str(d) for d in filtered_doc_ids)
        filtered_segments_dict = {seg_id: seg_data for seg_id, seg_data in segments_dict.items()
                                  if seg_id.split('/')[0] in _docs}

    filtered_model_dict = model_dict.copy()
    filtered_model_dict['segments_dict'] = filtered_segments_dict

    # Execute Boolean query
    try:
        executor = BooleanQueryExecutor(
            model_dict=filtered_model_dict,
            keyword_search_func=run_keyword_search_internal,
            scoring_strategy=config.boolean_scoring
        )
        results = executor.execute(ast)
        print(f'✓ Found {len(results)} matching segments')
    except Exception as e:
        print(f'❌ Query execution error: {e}')
        return [], {}

    if not results:
        return [], {}

    # Apply max results limit if set
    if config.max_results_per_topic is not None:
        results = results[:config.max_results_per_topic]

    # Create SearchResult objects
    documents_dict = model_dict['documents_dict']
    countries_dict = model_dict['countries_dict']

    search_results = []
    for segment_id, score in results:
        doc_id = segment_id.split('/')[0]

        # Get segment text
        segment_data = segments_dict.get(segment_id, {})
        text = get_segment_text_safe(segment_data) or ''

        # Get document metadata
        doc_data = documents_dict.get(doc_id, {})
        doc_name = doc_data.get('document', doc_data.get('name', ''))
        year = doc_data.get('year', 0)
        doc_type = doc_data.get('type', 'Unknown')

        # Get country information (matching keyword search pattern)
        country_iso = doc_data.get('iso', '')
        country_info = countries_dict.get(country_iso, {})
        country_name = country_info.get('country', doc_data.get('country', 'Unknown'))

        # Get context windows if configured
        context_before = ''
        context_after = ''
        if config.context_window > 0:
            context_before, context_after = get_context_window(
                segment_id, model_dict, config.context_window
            )

        # Create search result
        result = SearchResult(
            segment_id=segment_id,
            text=text,
            score=score,
            document_id=doc_id,
            document_name=doc_name,
            country=country_name,
            country_iso=country_iso,
            year=year,
            doc_type=doc_type,
            context_before=context_before,
            context_after=context_after
        )

        # Enrich with metadata
        result = enrich_result_with_metadata(result)

        search_results.append(result)

    search_results, _merged = _count_once(search_results, config)

    # Cluster results
    try:
        cluster_info = cluster_search_results_internal(
            search_results,
            model_dict,
            cluster_threshold=config.cluster_threshold,
            search_query=query
        )

        # Cluster names already assigned by cluster_search_results_internal
        print(f'✓ Clustered into {cluster_info.get("num_clusters", 0)} clusters '
              f'({cluster_info.get("num_singletons", 0)} singletons)')

    except Exception as e:
        print(f'⚠️  Clustering failed: {e}')
        print('Continuing with all results as singletons')
        cluster_info = {
            'cluster_dict': {},
            'cluster_labels': [-1] * len(search_results),
            'num_clusters': 0,
            'num_singletons': len(search_results),
            'cluster_names': {}
        }

    cluster_info['pairs_merged'] = _merged
    return search_results, cluster_info

print('✅ Boolean search functions defined')

In [ ]:
inject_app_favicon()

# Link helpers + clickable result-card override for Voila output

def _get_xhr_base_dir() -> Optional[Path]:
    candidates = []
    for base in [ROOT_PATH, CONFIGURED_INSTALL_PATH, ROOT_PATH / 'NSDDD_v3.5_workspace', ROOT_PATH / 'NSDDD_v3_workspace']:
        if base is None:
            continue
        try:
            candidate = Path(base).expanduser() / 'XHR'
        except Exception:
            continue
        candidates.append(candidate)

    for candidate in candidates:
        if candidate.exists():
            return candidate

    return candidates[0] if candidates else None


def get_segment_viewer_path(segment_id: str) -> Optional[Path]:
    parts = str(segment_id).split('/')
    if len(parts) == 3 and parts[1] == 'o':
        doc = parts[0] + '_original'
    elif len(parts) == 2:
        doc = parts[0]
    else:
        return None

    xhr_dir = _get_xhr_base_dir()
    if xhr_dir is None:
        return None

    return (xhr_dir / f"{doc}.html").resolve()


def get_segment_viewer_http_url(segment_id: str) -> str:
    parts = str(segment_id).split('/')
    if len(parts) == 3 and parts[1] == 'o':
        return f"/voila/files/XHR/{parts[0]}_original.html?segment={parts[2]}"
    if len(parts) == 2:
        return f"/voila/files/XHR/{parts[0]}.html?segment={parts[1]}"
    return ''


def get_segment_viewer_uri(segment_id: str) -> str:
    parts = str(segment_id).split('/')
    if len(parts) == 3 and parts[1] == 'o':
        document_id, segment_number = parts[0] + '_original', parts[2]
    elif len(parts) == 2:
        document_id, segment_number = parts[0], parts[1]
    else:
        return ''

    import os

    base_url = os.environ.get('NSDDD_VOILA_BASE_URL', 'http://127.0.0.1:8867').rstrip('/')
    return f"{base_url}/voila/files/XHR/{document_id}.html?segment={segment_number}"


def get_segment_viewer_link_html(segment_id: str, link_text: str = 'Open segment viewer') -> str:
    viewer_path = get_segment_viewer_path(segment_id)
    viewer_url = get_segment_viewer_http_url(segment_id)
    if viewer_path is None or not viewer_path.exists() or not viewer_url:
        return ''

    safe_text = escape(link_text)
    safe_url = escape(viewer_url)
    return f'<a href="{safe_url}" target="_blank" rel="noopener noreferrer" style="color:#1565c0;text-decoration:none;">{safe_text}</a>'


def _build_result_context_html(result: SearchResult) -> str:
    context_html = ''
    if result.context_before or result.context_after:
        context_html = '<div style="background-color: #f9f9f9; padding: 10px; margin: 8px 0; border-left: 3px solid #ccc; font-size: 12px; color: #666; font-style: italic;">'
        if result.context_before:
            cb = escape(result.context_before[:300]) + ('...' if len(result.context_before) > 300 else '')
            context_html += f'<p style="margin: 0 0 5px 0;"><strong>↑ Before:</strong> {cb}</p>'
        if result.context_after:
            ca = escape(result.context_after[:300]) + ('...' if len(result.context_after) > 300 else '')
            context_html += f'<p style="margin: 5px 0 0 0;"><strong>↓ After:</strong> {ca}</p>'
        context_html += '</div>'
    return context_html


def _num_tag(ids):
    nums = _seg_nums(ids)
    if not nums:
        return ''
    return ('<span style="font-family:ui-monospace,monospace;font-size:11px;color:#90a4ae;'
            'margin-right:5px;">[' + escape(', '.join(nums)) + ']</span>')


def _render_result_card(result: SearchResult, score_prefix: str) -> None:
    if result.cluster_name and result.cluster_name != 'Unclustered':
        _ex = getattr(result, 'cluster_example', '')
        cluster_display = (f' | <span class="nsddd-cl" tabindex="0" data-tip="{escape(_ex)}">🏷️ {escape(result.cluster_name)}</span>'
                           if _ex else f' | 🏷️ {escape(result.cluster_name)}')
    elif result.cluster_id is not None and result.cluster_id != 'singletons':
        cluster_display = f' | Cluster {escape(str(result.cluster_id))}'
    else:
        cluster_display = ''

    _seg_l = _seg_lang(result.segment_id)
    _other_txt, _other_label, _other_lang, _flag, _other_ids = _aligned_other(result.segment_id)
    if _flag == 'doc':
        _flag_html = '<div style="margin:6px 0 0 0;padding:4px 9px;border-left:3px solid #d97706;background:#fff7ed;color:#92400e;font-size:11px;">&#9888; Automatic alignment is unreliable for this document &mdash; original and English are not matched here. Use &ldquo;View side-by-side&rdquo; (independent columns) to compare.</div>'
    elif _flag == 'low':
        _flag_html = '<div style="margin:6px 0 0 0;padding:4px 9px;border-left:3px solid #d97706;background:#fff7ed;color:#92400e;font-size:11px;">&#9888; An original-language parallel exists, but the sentence alignment here is low-confidence &mdash; not shown. Use &ldquo;View side-by-side&rdquo; to compare.</div>'
    else:
        _flag_html = ''
    if _seg_l == 'en' and _other_txt:
        _primary_text, _badge_lang = _other_txt, _other_lang
        _counter_label, _counter_text = _english_label(str(result.segment_id).split('/')[0]), result.text
        _primary_ids, _counter_ids = _other_ids, [result.segment_id]
    else:
        _primary_text, _badge_lang = result.text, _seg_l
        _counter_label, _counter_text = _other_label, _other_txt
        _primary_ids, _counter_ids = [result.segment_id], _other_ids
    _aligned_html = ('<div dir="auto" style="margin:6px 0 0 0;padding-left:9px;border-left:3px solid #bfe3cf;color:#555;font-size:13px;">' + _num_tag(_counter_ids) + '<em>' + escape(_counter_label) + ':</em> ' + escape(_counter_text[:500]) + '</div>') if _counter_text else ''
    _bil_html = ''   # the parallel page is now the main link and is listed in the link row
    result_text = escape(_primary_text[:500]) + ('...' if len(_primary_text) > 500 else '')
    _main = segment_links(result.segment_id, absolute=False)['main']
    result_body = f'<a href="{escape(_main)}" target="_blank" rel="noopener noreferrer" style="color:#1565c0;text-decoration:none;">{result_text}</a>'
    result_link_html = segment_links_html(result.segment_id)

    display(HTML(f"""
    <div style="background-color: white; padding: 12px; margin: 8px 0 8px 20px;
                border-radius: 5px; border-left: 3px solid #9C27B0;">
        <p style="margin: 0 0 5px 0; color: #888; font-size: 11px;">
            {score_prefix} {_lang_badge(_badge_lang, str(result.segment_id).split('/')[0])}{cluster_display}
        </p>
        <p style="margin: 0 0 8px 0; color: #333; line-height: 1.6;">
            {_num_tag(_primary_ids)}{result_body}
        </p>
        {_aligned_html}
        {_flag_html}
        {result_link_html}
        {_bil_html}
        {_build_result_context_html(result)}
    </div>
    """))


def display_search_results(results: List[SearchResult], topic: str, filters: MetadataFilters,
                          config: SearchConfig):
    display(HTML(f"""
    <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                padding: 20px; border-radius: 10px; color: white; margin: 20px 0;">
        <h2 style="margin: 0; font-size: 24px;">🔍 Results for: "{escape(topic)}"</h2>
        <p style="margin: 10px 0 0 0; opacity: 0.9;">Found {len(results)} results in {len(set(r.document_id for r in results))} documents</p>
    </div>
    """))

    display_filter_summary(filters)

    search_type = 'Keyword Search' if config.keyword_search else 'Semantic Search'
    display(HTML(f"""
    <div style="background-color: #fff3e0; padding: 10px; margin: 10px 0;
                border-radius: 5px; border-left: 4px solid #ff9800;">
        <strong>⚙️ Search Configuration:</strong> {search_type} |
        Threshold: {config.search_threshold:.2f} |
        Cluster: {config.cluster_threshold:.2f}
    </div>
    """))

    doc_groups = {}
    for result in results:
        doc_groups.setdefault(result.document_id, []).append(result)

    for _doc_id, doc_results in sorted(doc_groups.items()):
        first_result = doc_results[0]
        metadata_tags = []
        if first_result.region:
            metadata_tags.append(f"🌍 {escape(first_result.region)}")
        if first_result.subregion:
            metadata_tags.append(f"📍 {escape(first_result.subregion)}")
        if first_result.income_group:
            metadata_tags.append(f"💰 {escape(first_result.income_group)}")
        if first_result.freedom_status:
            metadata_tags.append(f"🗳️ {escape(first_result.freedom_status)}")

        display(HTML(f"""
        <div style="background-color: #f5f5f5; padding: 15px; margin: 15px 0;
                    border-radius: 8px; border-left: 5px solid #4CAF50;">
            <h3 style="margin: 0 0 10px 0; color: #546e7a;">
                📄 {escape(first_result.country)} ({first_result.year})
            </h3>
            <p style="margin: 5px 0; color: #555; font-size: 14px;">
                <strong>{escape(first_result.document_name)}</strong>
            </p>
            <p style="margin: 5px 0; color: #777; font-size: 12px;">
                {' | '.join(metadata_tags)}
            </p>
            <p style="margin: 5px 0; color: #999; font-size: 12px;">
                📊 {len(doc_results)} results in this document
            </p>
        </div>
        """))

        for i, result in enumerate(doc_results[:5], 1):
            _render_result_card(result, f'Result {i} | Score: {result.score:.3f}')

        if len(doc_results) > 5:
            display(HTML(f"""
            <p style="margin: 5px 0 5px 20px; color: #999; font-size: 12px; font-style: italic;">
                ... and {len(doc_results) - 5} more results
            </p>
            """))


def display_batch_results(all_results: Dict[str, List[SearchResult]], all_cluster_info: Dict,
                         config: SearchConfig, max_per_topic: int = 5):
    doc_groups = {}

    for topic, results in all_results.items():
        topic_safe = escape(topic)
        if LANG_VIEW != 'all':
            results = [r for r in results if ((_seg_lang(r.segment_id) == 'en') if LANG_VIEW == 'en' else (_seg_lang(r.segment_id) != 'en'))]
        if not results:
            display(HTML(f"""
            <div style="background-color: #ffebee; padding: 15px; margin: 20px 0;
                        border-radius: 8px; border-left: 4px solid #f44336;">
                <strong>❌ No results found for: "{topic_safe}"</strong>
            </div>
            """))
            continue

        cluster_info = all_cluster_info.get(topic, {})
        _enc, _oc = _en_orig_counts(results)
        _bd = f' ({_enc} English + {_oc} original-language)' if _oc else ''
        _pm = cluster_info.get('pairs_merged', 0)
        if _pm:
            _bd += f' &middot; {_pm} original-language match' + ('es' if _pm != 1 else '') + ' merged into the English match of the same passage'
        num_clusters = cluster_info.get('num_clusters', 0)
        num_singletons = cluster_info.get('num_singletons', 0)
        search_type = 'Boolean Search' if config.boolean_mode else ('Keyword Search' if config.keyword_search else 'Semantic Search')

        display(HTML(f"""
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    padding: 20px; border-radius: 10px; color: white; margin: 20px 0;">
            <h2 style="margin: 0; font-size: 24px;">🔍 {escape(search_type)} Results: "{topic_safe}"</h2>
            <p style="margin: 10px 0 0 0; opacity: 0.9;">
                Found {len(results)} results in {len({r.document_id for r in results})} documents{_bd} |
                {num_clusters} clusters + {num_singletons} singletons
            </p>
        </div>
        """))

        results = _collapse_pairs(results)
        _n_docs = len({r.document_id for r in results})     # documents with results (all of them, not only the preview)
        doc_groups.clear()
        for result in results[:max_per_topic * 5]:
            doc_groups.setdefault(result.document_id, []).append(result)

        for _doc_id, doc_results in list(doc_groups.items())[:max_per_topic]:
            first_result = doc_results[0]
            metadata_tags = []
            if first_result.region:
                metadata_tags.append(f'🌍 {escape(first_result.region)}')
            if first_result.subregion:
                metadata_tags.append(f'📍 {escape(first_result.subregion)}')
            if first_result.income_group:
                metadata_tags.append(f'💰 {escape(first_result.income_group)}')
            if first_result.freedom_status:
                metadata_tags.append(f'🗳️ {escape(first_result.freedom_status)}')

            metadata_str = ' | '.join(metadata_tags)

            display(HTML(f"""
            <div style="background-color: #f5f5f5; padding: 15px; margin: 15px 0;
                        border-radius: 8px; border-left: 5px solid #4CAF50;">
                <h3 style="margin: 0 0 10px 0; color: #546e7a;">
                    📄 {escape(first_result.country)} ({first_result.year})
                </h3>
                <p style="margin: 5px 0; color: #555; font-size: 14px;">
                    <strong>{escape(first_result.document_name)}</strong>
                </p>
                <p style="margin: 5px 0; color: #777; font-size: 12px;">
                    {metadata_str}
                </p>
                <p style="margin: 5px 0; color: #999; font-size: 12px;">
                    📊 {len(doc_results)} results in this document
                </p>
            </div>
            """))

            for result in doc_results[:3]:
                score_indicator = '🎯' if result.score > 0.8 else '✅' if result.score > 0.65 else '📍'
                _render_result_card(result, f'{score_indicator} Score: {result.score:.3f}')

            if len(doc_results) > 3:
                display(HTML(f"""
                <p style="margin: 5px 0 5px 20px; color: #999; font-size: 12px; font-style: italic;">
                    ... and {len(doc_results) - 3} more results in this document
                </p>
                """))

        if _n_docs > min(max_per_topic, len(doc_groups)):
            # the count was taken from the 25 previewed results only (e.g. '5 of 22'), not from all results
            display(HTML(f"""
            <p style="margin: 10px 0; color: #666; font-size: 12px; font-style: italic;">
                ... showing the top {min(max_per_topic, len(doc_groups))} of {_n_docs} documents with results (see the exported CSV for all)
            </p>
            """))




In [ ]:
import ipywidgets as _w
_loading_widget = _w.HTML(value=
    '<div style="padding:20px;font-family:sans-serif;font-size:1.1em;color:#555;">'
    '&#9203; <strong>Loading the search model &mdash; this takes up to a minute.</strong><br>'
    '<span style="font-size:0.9em;">Please leave this window open. '
    'The search interface will appear automatically when ready.</span></div>'
)
display(_loading_widget)


In [ ]:
%%capture
# Initialize model (optional for the MCP and API servers, which load it themselves; the search page loads it here when it opens).
# Load the data and the semantic encoder when the page opens, so the first search is as fast as later ones.
# Low Power mode loads the encoder on the first semantic search instead (keyword and Boolean searches never need it).
MODEL_DICT, ENCODER = initialize_model(load_encoder=not LOW_POWER)

In [ ]:
_loading_widget.value = ''
_update_banner_html = build_update_banner_html()
if _update_banner_html:
    display(HTML(_update_banner_html))

display(HTML(
    '<div style="font-family:sans-serif;font-size:0.92em;line-height:1.5;background:#fff8e1;border-left:4px solid #f9a825;'
    'color:#1a202c;padding:8px 14px;margin:6px 0 12px 0;border-radius:4px;">'
    '<strong>Pre-release version.</strong> You are using a pre-release of NSDDD 4.0. The full 4.0 dataset will be released '
    'early in the new year, with all the new documents from 2026 plus other additions as they are found. Bug fixes are '
    'published as updates in the meantime. Documents, figures and results may change before the full release.</div>'))

interface = create_search_interface()
display(interface)

from IPython.display import Javascript as _JS
display(_JS('''
(function() {
    var lastN = 0;
    setInterval(function() {
        var t = document.getElementById('ns-scroll-trigger');
        if (!t) return;
        var n = parseInt(t.getAttribute('data-n') || '0', 10);
        if (n > lastN) {
            lastN = n;
            setTimeout(function() {
                var el = document.getElementById('search-output-anchor');
                if (el) { el.scrollIntoView({behavior:'smooth'}); window.scrollBy(0,120); }
            }, 300);
        }
    }, 250);
})();
'''))
